n this experiment we do a short manual hyperparamter search of PC-kNN for VIT on a transformed imagenet subset and then evalute the best hyperparamter on SI-SCORE. We als ocreate a trade of figure when gating is used.

In [ ]:
import copy

import torch
import torchvision
from matplotlib import pyplot as plt


from search.coordinate_descent import WeightedCoordinateDescent
from src.utils.eval.vis import plt_setup_latex

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
#look for experiment files in parents
import os
path_found = False
current_path = os.getcwd()
while not path_found:
    if os.path.exists(os.path.join(current_path, "experiment_files")):
        path_found = True
        break
    current_path = os.path.dirname(current_path)


In [ ]:
experiment_files_path_data = os.path.join(current_path, "experiment_files", "data")

In [ ]:
dataset ="si_score"
architecture = "vit_b_16_pretrained"

In [ ]:

from data.get_dataset import get_dataset, get_dataset_info

dataset_info = get_dataset_info(dataset)
dataset_dict = get_dataset(dataset_info,path=experiment_files_path_data)


In [ ]:
dataset_train = dataset_dict['train_dataset']
dataset_val = dataset_dict['val_dataset']
dataset_test = dataset_dict['test_dataset']
train_loader = dataset_dict['train_loader']
val_loader = dataset_dict['val_loader']
test_loader = dataset_dict['test_loader']
n_classes = dataset_info.num_classes
train_loader_transformed = dataset_dict['train_loader_transformed']
val_loader_transformed = dataset_dict['val_loader_transformed']
test_loader_transformed = dataset_dict['test_loader_transformed']
train_loader_no_shuffle = dataset_dict['train_loader_no_shuffle']



In [ ]:
print(len(dataset_train))

In [ ]:
print(len(train_loader))


In [ ]:
dataset_train[2][0].shape

In [ ]:
batch_size = next(iter(train_loader))[0].shape[0]

In [ ]:
#test images
fig, axs = plt.subplots(nrows=3, ncols=1, figsize=(10, 5))

axs[0].imshow(torchvision.utils.make_grid(next(iter(train_loader))[0], nrow=batch_size//4).permute(1, 2, 0).cpu())
axs[1].imshow(torchvision.utils.make_grid(next(iter(val_loader))[0], nrow=batch_size//4).permute(1, 2, 0).cpu())
axs[2].imshow(torchvision.utils.make_grid(next(iter(test_loader_transformed))[0], nrow=batch_size//4).permute(1, 2, 0).cpu())
axs[0].set_title('Training')
axs[1].set_title('Validation')
axs[2].set_title('Test')



In [ ]:

from model.train import train_and_get_model
from model.get_model import get_network
from src.utils.eval.main_model import evaluate_base_model

model_dir_path = os.path.join(current_path, "experiment_files", "models")
embedding_cache_path = os.path.join(current_path, "experiment_files", "embedding_cache")

model = get_network(dataset_info,architecture, num_classes=n_classes).to(device)
modelname = f"{dataset}_{architecture}"
cache_name_train= f"{dataset}_{architecture}_embedding_cache_train"

#if si score then no train
if not "si_score" in dataset:
    train_and_get_model(model,model_dir_path,modelname, train_loader, val_loader , trainer_kwargs= {
            "accelerator": "auto",
            "max_epochs": 100,
            "precision": "16-mixed",
    },load_if_exists=True)


#res = evaluate_base_model(model, test_loader_transformed, device)
#print(res)

In [ ]:
#check main model
res = evaluate_base_model(model, test_loader_transformed, device)
print(res)
res = evaluate_base_model(model, test_loader, device)
print(res)

In [ ]:
import torch
import torch.nn as nn
import numpy as np

class TestTimeAugmenter(nn.Module):
    """
    Wraps a model to perform Test-Time Augmentation (TTA).
    Uses a systematic linspace for cyclical rotations.
    """
    def __init__(self, model, transformation_problem, n_samples=16):
        super().__init__()
        self.model = model
        self.transformation_problem = transformation_problem
        self.n_samples = n_samples

    def forward(self, x):
        if self.training:
            return self.model(x)

        batch_size = x.shape[0]
        device = x.device

        combined_logits = self.model(x)


        angles = np.linspace(-np.pi, np.pi, self.n_samples, endpoint=False)
        if self.n_samples %2 ==0:
            zero_idx = np.abs(angles).argmin()
            assert -1e-8<np.abs(angles).min()<1e-8
            augmented_angles = np.delete(angles, zero_idx)

        for angle in augmented_angles:
            params = torch.full((batch_size, 1), angle, device=device)
            x_aug = self.transformation_problem.transform(x, params)
            combined_logits += self.model(x_aug)

        return combined_logits / self.n_samples

In [ ]:
model_dir_path = os.path.join(current_path, "experiment_files", "models")
embedding_cache_path = os.path.join(current_path, "experiment_files", "embedding_cache")
# Add results dir and helper for save paths
results_dir_path = os.path.join(current_path, "experiment_files", "results", dataset, architecture, "comparison_siscore_vit")
os.makedirs(results_dir_path, exist_ok=True)

transform_name = "rotation"


def savepath(label: str) -> str:
    safe = "".join(c if c.isalnum() or c in "-_." else "_" for c in label)
    return os.path.join(results_dir_path,transform_name, f"{safe}.json")

In [ ]:
import confidence.direct.logit_based
import src.utils.affine_transforms

energy = confidence.direct.logit_based.EnergyConfidence()
from src.utils.transform_sequence import TransformSequence

transformation_sequence = TransformSequence(
    transformations=[
        src.utils.affine_transforms.AffineTransformation2D.ROTATION.value,
    ],
    domains=[(-torch.pi, torch.pi)],
    device=device,init_method="sobol"
)


In [ ]:
model_test_time = TestTimeAugmenter(model,transformation_sequence,n_samples=16)


In [ ]:
import json, os

# 1. Transformed Model
transformed_path = savepath("result_tta_transformed")
# Ensure the directory exists before saving
os.makedirs(os.path.dirname(transformed_path), exist_ok=True)

if not os.path.exists(transformed_path):
    res = evaluate_base_model(model_test_time, test_loader_transformed, device)
    json.dump(res, open(transformed_path, 'w'))
print(json.load(open(transformed_path)))

# 2. Raw Model
raw_path = savepath("result_tta")
# Ensure the directory exists before saving
os.makedirs(os.path.dirname(raw_path), exist_ok=True)

if not os.path.exists(raw_path):
    res = evaluate_base_model(model_test_time, test_loader, device)
    json.dump(res, open(raw_path, 'w'))
print(json.load(open(raw_path)))

In [ ]:
test_loader_transformed = torch.utils.data.DataLoader(test_loader_transformed.dataset, batch_size=8, shuffle=False, num_workers=20,pin_memory=True,persistent_workers=True,prefetch_factor=4)

In [ ]:
transform_seq = transformation_sequence.cuda()

In [ ]:
from get_dataset import get_layer_embedding_cache_config, create_layer_embedding_cache

cache_config = get_layer_embedding_cache_config(dataset, architecture,transform_name=None,dataset_info=dataset_info)
train_cache =create_layer_embedding_cache(model, train_loader_no_shuffle,cache_config, embedding_cache_path, device=device)

In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
print(list(model.named_modules()))

In [ ]:
model(next(iter(train_loader_no_shuffle))[0].to(device)).max().cpu().detach().numpy()

In [ ]:

from model.get_model import get_network_layer

layer,layer_io = get_network_layer(dataset_info, architecture, 0)

embeddings_t, final_t, classes_t = train_cache.get_correct_embeddings(layer, capture_modes=layer_io, flatten=True)
dual_output_model = train_cache.make_wrapper(layer, capture_modes=layer_io, concat=False, flatten=True)


In [ ]:
cache_train = train_cache

In [ ]:
from search.objective_generators import WCD_LATTICE_WRAPPER

wcd = WeightedCoordinateDescent(
    rounds=1,samples_per_dim=[17,])

di = WCD_LATTICE_WRAPPER(wcd)
wcd = None

In [ ]:

embeddings_t, final_t, classes_t = cache_train.get_correct_embeddings(layer, capture_modes=layer_io, flatten=True, concat=True)
dual_output_model = cache_train.make_wrapper(layer, capture_modes=layer_io, concat=True, flatten=True)

from src.utils.eval.ood_performance import evaluate_confidence_module, evaluate_confidence_and_search

from src.utils.transformation_problem import TransformationProblem
from confidence.model.single_pass import SinglePassConfidence
from confidence.direct.logit_based import EnergyConfidence
from confidence.control.split import SplitConfidence, PredictedSplitConfidence
from confidence.unsupervised.classic.nn_pytorch import KNNConfidence, PerClassKNNConfidence

from confidence.input_transform import InputTransformImage, PCAInputModule, RandomProjectionModule



nn_pytorch_pretrained = PerClassKNNConfidence(metric="cosine", input_transform=None)
nn_pytorch_pretrained.fit(embeddings_t, classes_t)
nn_pytorch_pretrained.cuda()

conf_split_pretrained = PredictedSplitConfidence(nn_pytorch_pretrained, EnergyConfidence(), mult=False, b=0.000)
conf_mod_nn_pytorch_pretrained = SinglePassConfidence(dual_output_model, conf_split_pretrained, index=1)
problem_nn_pytorch_pretrained = TransformationProblem(conf_mod_nn_pytorch_pretrained, transform_seq,
                                                      consolidate_method="consolidate_simple",max_batch_size=32)
model.cuda().eval()

from src.utils.eval.ood_performance import load_or_run_evaluate_confidence_and_search

load_or_run_evaluate_confidence_and_search(
    model, optimizer=di, problem=problem_nn_pytorch_pretrained,
    test_loader=val_loader_transformed, max_batch_override=dataset_info.batch_size_search,
    save_path=savepath("pcknn_layer_0"),show_progress=True,
    repeats=1)




In [ ]:
energy_confidence = SinglePassConfidence(model, EnergyConfidence())
from src.utils.eval.ood_performance import load_or_run_evaluate_confidence_and_search

load_or_run_evaluate_confidence_and_search(
    model, optimizer=di, problem=TransformationProblem(energy_confidence, transform_seq,
                                                                  consolidate_method="consolidate_simple"),
    test_loader=val_loader_transformed, max_batch_override=dataset_info.batch_size_search,
    save_path=savepath("normal_energy_val"),show_progress=True,
    repeats=1)

In [ ]:
for name, _ in model.named_modules():
    print(name)


In [ ]:

layer,layer_io = get_network_layer(dataset_info, architecture, 1)

embeddings_t, final_t, classes_t = train_cache.get_correct_embeddings(layer, capture_modes=layer_io, flatten=True)
dual_output_model = train_cache.make_wrapper(layer, capture_modes=layer_io, concat=False, flatten=True)


In [ ]:
print()
embeddings_t, final_t, classes_t = cache_train.get_correct_embeddings(layer, capture_modes=layer_io, flatten=True, concat=True)
dual_output_model = cache_train.make_wrapper(layer, capture_modes=layer_io, concat=True, flatten=True)

from src.utils.eval.ood_performance import evaluate_confidence_module, evaluate_confidence_and_search

from src.utils.transformation_problem import TransformationProblem
from confidence.model.single_pass import SinglePassConfidence
from confidence.direct.logit_based import EnergyConfidence
from confidence.control.split import SplitConfidence, PredictedSplitConfidence
from confidence.unsupervised.classic.nn_pytorch import KNNConfidence, PerClassKNNConfidence

from confidence.input_transform import InputTransformImage, PCAInputModule, RandomProjectionModule



nn_pytorch_pretrained = PerClassKNNConfidence(metric="cosine", input_transform=None)
nn_pytorch_pretrained.fit(embeddings_t, classes_t)
nn_pytorch_pretrained.cuda()

conf_split_pretrained = PredictedSplitConfidence(nn_pytorch_pretrained, EnergyConfidence(), mult=False, b=0.000)
conf_mod_nn_pytorch_pretrained = SinglePassConfidence(dual_output_model, conf_split_pretrained, index=1)
problem_nn_pytorch_pretrained = TransformationProblem(conf_mod_nn_pytorch_pretrained, transform_seq,
                                                      consolidate_method="consolidate_simple",max_batch_size=32)
model.cuda().eval()

print()
load_or_run_evaluate_confidence_and_search(
    model, optimizer=di, problem=problem_nn_pytorch_pretrained,
    test_loader=val_loader_transformed, max_batch_override=dataset_info.batch_size_search,
    save_path=savepath("pcknn_layer_1"),show_progress=True,
    repeats=1)

In [ ]:

embeddings_t, final_t, classes_t = cache_train.get_correct_embeddings(layer, capture_modes=layer_io, flatten=True, concat=True)
dual_output_model = cache_train.make_wrapper(layer, capture_modes=layer_io, concat=True, flatten=True)

from src.utils.eval.ood_performance import evaluate_confidence_module, evaluate_confidence_and_search

from src.utils.transformation_problem import TransformationProblem
from confidence.model.single_pass import SinglePassConfidence
from confidence.direct.logit_based import EnergyConfidence
from confidence.control.split import SplitConfidence, PredictedSplitConfidence
from confidence.unsupervised.classic.nn_pytorch import KNNConfidence, PerClassKNNConfidence

from confidence.input_transform import InputTransformImage, PCAInputModule, RandomProjectionModule



nn_pytorch_pretrained = PerClassKNNConfidence(metric="mixed", input_transform=None,mixed_alpha=0.5)
nn_pytorch_pretrained.fit(embeddings_t, classes_t)
nn_pytorch_pretrained.cuda()

conf_split_pretrained = PredictedSplitConfidence(nn_pytorch_pretrained, EnergyConfidence(), mult=False, b=0.000)
conf_mod_nn_pytorch_pretrained = SinglePassConfidence(dual_output_model, conf_split_pretrained, index=1)
problem_nn_pytorch_pretrained = TransformationProblem(conf_mod_nn_pytorch_pretrained, transform_seq,
                                                      consolidate_method="consolidate_simple",max_batch_size=32)
model.cuda().eval()

from src.utils.eval.ood_performance import load_or_run_evaluate_confidence_and_search

load_or_run_evaluate_confidence_and_search(
    model, optimizer=di, problem=problem_nn_pytorch_pretrained,
    test_loader=val_loader_transformed, max_batch_override=dataset_info.batch_size_search,
    save_path=savepath("pcknn_layer_0_mixed"),show_progress=True,
    repeats=1)




In [ ]:

embeddings_t, final_t, classes_t = cache_train.get_correct_embeddings(layer, capture_modes=layer_io, flatten=True, concat=True)
dual_output_model = cache_train.make_wrapper(layer, capture_modes=layer_io, concat=True, flatten=True)

from src.utils.eval.ood_performance import evaluate_confidence_module, evaluate_confidence_and_search

from src.utils.transformation_problem import TransformationProblem
from confidence.model.single_pass import SinglePassConfidence
from confidence.direct.logit_based import EnergyConfidence
from confidence.control.split import SplitConfidence, PredictedSplitConfidence
from confidence.unsupervised.classic.nn_pytorch import KNNConfidence, PerClassKNNConfidence

from confidence.input_transform import InputTransformImage, PCAInputModule, RandomProjectionModule


nn_pytorch_pretrained = PerClassKNNConfidence(metric="mixed", input_transform=None,mixed_alpha=0.5)
nn_pytorch_pretrained.fit(embeddings_t, classes_t)
nn_pytorch_pretrained.cuda()

conf_split_pretrained = PredictedSplitConfidence(nn_pytorch_pretrained, EnergyConfidence(), mult=False, b=0.000)
conf_mod_nn_pytorch_pretrained = SinglePassConfidence(dual_output_model, conf_split_pretrained, index=1)
problem_nn_pytorch_pretrained = TransformationProblem(conf_mod_nn_pytorch_pretrained, transform_seq,
                                                      consolidate_method="consolidate_simple",max_batch_size=32)
model.cuda().eval()

from src.utils.eval.ood_performance import load_or_run_evaluate_confidence_and_search

load_or_run_evaluate_confidence_and_search(
    model, optimizer=di, problem=problem_nn_pytorch_pretrained,
    test_loader=test_loader_transformed, max_batch_override=dataset_info.batch_size_search,
    save_path=savepath("final_results_mixed"),show_progress=True,
    repeats=1)




In [ ]:
load_or_run_evaluate_confidence_and_search(
    model, optimizer=di, problem=problem_nn_pytorch_pretrained,
    test_loader=test_loader_transformed, max_batch_override=dataset_info.batch_size_search,
    save_path=savepath("final_results_mixed"),show_progress=True,
    repeats=2)

In [ ]:
load_or_run_evaluate_confidence_and_search(
    model, optimizer=di, problem=problem_nn_pytorch_pretrained,
    test_loader=test_loader_transformed, max_batch_override=dataset_info.batch_size_search,
    save_path=savepath("final_results_mixed"),show_progress=True,
    repeats=3)

In [ ]:
load_or_run_evaluate_confidence_and_search(
    model, optimizer=di, problem=problem_nn_pytorch_pretrained,
    test_loader=test_loader_transformed, max_batch_override=dataset_info.batch_size_search,
    save_path=savepath("final_results_mixed"),show_progress=True,
    repeats=4)

In [ ]:
import os
from typing import Optional
import matplotlib.pyplot as plt
import torch
import torchvision.transforms.functional as TF

@torch.no_grad()
def visualize_rotation_optimization(
    model,
    optimizer,
    problem,
    test_loader,
    class_names,  # List mapping label indices to actual class names
    confidence_module=None,
    num_rotations: int = 8,
    sample_idx: int = 0,
    device: Optional[torch.device] = None,
    figsize=(20, 8),
    save_dir: Optional[str] = None  # If None, do not save
):
    """
    Visualize optimizer behavior and optionally save info about all images in two text files:
    - rotated_info.txt
    - transformed_info.txt
    """
    if device is None:
        device = next(model.parameters()).device

    if save_dir is not None:
        os.makedirs(save_dir, exist_ok=True)

    model.eval()
    if confidence_module is not None:
        confidence_module.eval()

    # Get sample image
    data_batch, label_batch = test_loader.dataset[sample_idx]
    original_img = data_batch.unsqueeze(0).to(device)
    true_label = label_batch
    true_class = class_names[true_label]

    # Generate rotations
    angles = torch.linspace(-180, 180, steps=num_rotations+1)[:-1]  # Avoid duplicate at 180°
    rotated_images = []
    transformed_images = []
    predictions_before = []
    confidences_before = []
    predictions_after = []
    confidences_after = []

    for angle in angles:
        rotated = TF.rotate(original_img, angle.item())
        rotated_images.append(rotated)

        # Prediction & confidence before optimization
        logits_rot = model(rotated)
        pred_rot = logits_rot.argmax(dim=-1).item()
        conf_rot_val = float(logits_rot.softmax(dim=-1).max())
        if confidence_module is not None:
            conf_rot_val, _ = confidence_module(rotated)
            conf_rot_val = conf_rot_val.item()

        predictions_before.append(pred_rot)
        confidences_before.append(conf_rot_val)

        # Optimize transformation
        best_param, _, _ = optimizer.optimize(problem, rotated, y=None)
        x_trans = problem.transform(rotated, best_param)
        transformed_images.append(x_trans)

        # Prediction & confidence after optimization
        logits_trans = model(x_trans)
        pred_trans = logits_trans.argmax(dim=-1).item()
        conf_trans_val = float(logits_trans.softmax(dim=-1).max())
        if confidence_module is not None:
            conf_trans_val, _ = confidence_module(x_trans)
            conf_trans_val = conf_trans_val.item()

        predictions_after.append(pred_trans)
        confidences_after.append(conf_trans_val)

    # Save text files with all info
    if save_dir is not None:
        rotated_txt_path = os.path.join(save_dir, "rotated_info.txt")
        transformed_txt_path = os.path.join(save_dir, "transformed_info.txt")

        with open(rotated_txt_path, "w") as f_rot, open(transformed_txt_path, "w") as f_trans:
            for i, angle in enumerate(angles):
                f_rot.write(
                    f"Image {i} | True class: {true_class} | Rotation angle: {angle:.1f}° | "
                    f"Predicted class: {class_names[predictions_before[i]]} | "
                    f"Confidence: {confidences_before[i]:.4f}\n"
                )
                f_trans.write(
                    f"Image {i} | True class: {true_class} | "
                    f"Predicted class: {class_names[predictions_after[i]]} | "
                    f"Confidence: {confidences_after[i]:.4f}\n"
                )

    # Optional visualization
    fig, axes = plt.subplots(2, num_rotations, figsize=figsize)
    for i in range(num_rotations):
        # Rotated
        img_disp = rotated_images[i].squeeze(0).cpu()
        if img_disp.shape[0] == 3:
            img_disp = img_disp.permute(1, 2, 0)
        img_disp = (img_disp - img_disp.min()) / (img_disp.max() - img_disp.min())
        axes[0, i].imshow(img_disp)
        axes[0, i].set_title(f"{class_names[predictions_before[i]]}\n{confidences_before[i]:.3f}",
                             color='green' if predictions_before[i] == true_label else 'red', fontsize=10)
        axes[0, i].axis('off')

        # Transformed
        trans_disp = transformed_images[i].squeeze(0).cpu()
        if trans_disp.shape[0] == 3:
            trans_disp = trans_disp.permute(1, 2, 0)
        trans_disp = (trans_disp - trans_disp.min()) / (trans_disp.max() - trans_disp.min())
        axes[1, i].imshow(trans_disp)
        axes[1, i].set_title(f"{class_names[predictions_after[i]]}\n{confidences_after[i]:.3f}",
                             color='green' if predictions_after[i] == true_label else 'red', fontsize=10)
        axes[1, i].axis('off')

    fig.suptitle(f"True Class: {true_class} | Rotation Optimization Analysis", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    return {
        "true_label": true_label,
        "true_class": true_class,
        "angles": angles.tolist(),
        "predictions_before": predictions_before,
        "confidences_before": confidences_before,
        "predictions_after": predictions_after,
        "confidences_after": confidences_after,
        "correct_rate_after": sum(p == true_label for p in predictions_after) / num_rotations
    }


In [ ]:
class_map = {0: 'tench, Tinca tinca',
 1: 'goldfish, Carassius auratus',
 2: 'great white shark, white shark, man-eater, man-eating shark, Carcharodon carcharias',
 3: 'tiger shark, Galeocerdo cuvieri',
 4: 'hammerhead, hammerhead shark',
 5: 'electric ray, crampfish, numbfish, torpedo',
 6: 'stingray',
 7: 'cock',
 8: 'hen',
 9: 'ostrich, Struthio camelus',
 10: 'brambling, Fringilla montifringilla',
 11: 'goldfinch, Carduelis carduelis',
 12: 'house finch, linnet, Carpodacus mexicanus',
 13: 'junco, snowbird',
 14: 'indigo bunting, indigo finch, indigo bird, Passerina cyanea',
 15: 'robin, American robin, Turdus migratorius',
 16: 'bulbul',
 17: 'jay',
 18: 'magpie',
 19: 'chickadee',
 20: 'water ouzel, dipper',
 21: 'kite',
 22: 'bald eagle, American eagle, Haliaeetus leucocephalus',
 23: 'vulture',
 24: 'great grey owl, great gray owl, Strix nebulosa',
 25: 'European fire salamander, Salamandra salamandra',
 26: 'common newt, Triturus vulgaris',
 27: 'eft',
 28: 'spotted salamander, Ambystoma maculatum',
 29: 'axolotl, mud puppy, Ambystoma mexicanum',
 30: 'bullfrog, Rana catesbeiana',
 31: 'tree frog, tree-frog',
 32: 'tailed frog, bell toad, ribbed toad, tailed toad, Ascaphus trui',
 33: 'loggerhead, loggerhead turtle, Caretta caretta',
 34: 'leatherback turtle, leatherback, leathery turtle, Dermochelys coriacea',
 35: 'mud turtle',
 36: 'terrapin',
 37: 'box turtle, box tortoise',
 38: 'banded gecko',
 39: 'common iguana, iguana, Iguana iguana',
 40: 'American chameleon, anole, Anolis carolinensis',
 41: 'whiptail, whiptail lizard',
 42: 'agama',
 43: 'frilled lizard, Chlamydosaurus kingi',
 44: 'alligator lizard',
 45: 'Gila monster, Heloderma suspectum',
 46: 'green lizard, Lacerta viridis',
 47: 'African chameleon, Chamaeleo chamaeleon',
 48: 'Komodo dragon, Komodo lizard, dragon lizard, giant lizard, Varanus komodoensis',
 49: 'African crocodile, Nile crocodile, Crocodylus niloticus',
 50: 'American alligator, Alligator mississipiensis',
 51: 'triceratops',
 52: 'thunder snake, worm snake, Carphophis amoenus',
 53: 'ringneck snake, ring-necked snake, ring snake',
 54: 'hognose snake, puff adder, sand viper',
 55: 'green snake, grass snake',
 56: 'king snake, kingsnake',
 57: 'garter snake, grass snake',
 58: 'water snake',
 59: 'vine snake',
 60: 'night snake, Hypsiglena torquata',
 61: 'boa constrictor, Constrictor constrictor',
 62: 'rock python, rock snake, Python sebae',
 63: 'Indian cobra, Naja naja',
 64: 'green mamba',
 65: 'sea snake',
 66: 'horned viper, cerastes, sand viper, horned asp, Cerastes cornutus',
 67: 'diamondback, diamondback rattlesnake, Crotalus adamanteus',
 68: 'sidewinder, horned rattlesnake, Crotalus cerastes',
 69: 'trilobite',
 70: 'harvestman, daddy longlegs, Phalangium opilio',
 71: 'scorpion',
 72: 'black and gold garden spider, Argiope aurantia',
 73: 'barn spider, Araneus cavaticus',
 74: 'garden spider, Aranea diademata',
 75: 'black widow, Latrodectus mactans',
 76: 'tarantula',
 77: 'wolf spider, hunting spider',
 78: 'tick',
 79: 'centipede',
 80: 'black grouse',
 81: 'ptarmigan',
 82: 'ruffed grouse, partridge, Bonasa umbellus',
 83: 'prairie chicken, prairie grouse, prairie fowl',
 84: 'peacock',
 85: 'quail',
 86: 'partridge',
 87: 'African grey, African gray, Psittacus erithacus',
 88: 'macaw',
 89: 'sulphur-crested cockatoo, Kakatoe galerita, Cacatua galerita',
 90: 'lorikeet',
 91: 'coucal',
 92: 'bee eater',
 93: 'hornbill',
 94: 'hummingbird',
 95: 'jacamar',
 96: 'toucan',
 97: 'drake',
 98: 'red-breasted merganser, Mergus serrator',
 99: 'goose',
 100: 'black swan, Cygnus atratus',
 101: 'tusker',
 102: 'echidna, spiny anteater, anteater',
 103: 'platypus, duckbill, duckbilled platypus, duck-billed platypus, Ornithorhynchus anatinus',
 104: 'wallaby, brush kangaroo',
 105: 'koala, koala bear, kangaroo bear, native bear, Phascolarctos cinereus',
 106: 'wombat',
 107: 'jellyfish',
 108: 'sea anemone, anemone',
 109: 'brain coral',
 110: 'flatworm, platyhelminth',
 111: 'nematode, nematode worm, roundworm',
 112: 'conch',
 113: 'snail',
 114: 'slug',
 115: 'sea slug, nudibranch',
 116: 'chiton, coat-of-mail shell, sea cradle, polyplacophore',
 117: 'chambered nautilus, pearly nautilus, nautilus',
 118: 'Dungeness crab, Cancer magister',
 119: 'rock crab, Cancer irroratus',
 120: 'fiddler crab',
 121: 'king crab, Alaska crab, Alaskan king crab, Alaska king crab, Paralithodes camtschatica',
 122: 'American lobster, Northern lobster, Maine lobster, Homarus americanus',
 123: 'spiny lobster, langouste, rock lobster, crawfish, crayfish, sea crawfish',
 124: 'crayfish, crawfish, crawdad, crawdaddy',
 125: 'hermit crab',
 126: 'isopod',
 127: 'white stork, Ciconia ciconia',
 128: 'black stork, Ciconia nigra',
 129: 'spoonbill',
 130: 'flamingo',
 131: 'little blue heron, Egretta caerulea',
 132: 'American egret, great white heron, Egretta albus',
 133: 'bittern',
 134: 'crane',
 135: 'limpkin, Aramus pictus',
 136: 'European gallinule, Porphyrio porphyrio',
 137: 'American coot, marsh hen, mud hen, water hen, Fulica americana',
 138: 'bustard',
 139: 'ruddy turnstone, Arenaria interpres',
 140: 'red-backed sandpiper, dunlin, Erolia alpina',
 141: 'redshank, Tringa totanus',
 142: 'dowitcher',
 143: 'oystercatcher, oyster catcher',
 144: 'pelican',
 145: 'king penguin, Aptenodytes patagonica',
 146: 'albatross, mollymawk',
 147: 'grey whale, gray whale, devilfish, Eschrichtius gibbosus, Eschrichtius robustus',
 148: 'killer whale, killer, orca, grampus, sea wolf, Orcinus orca',
 149: 'dugong, Dugong dugon',
 150: 'sea lion',
 151: 'Chihuahua',
 152: 'Japanese spaniel',
 153: 'Maltese dog, Maltese terrier, Maltese',
 154: 'Pekinese, Pekingese, Peke',
 155: 'Shih-Tzu',
 156: 'Blenheim spaniel',
 157: 'papillon',
 158: 'toy terrier',
 159: 'Rhodesian ridgeback',
 160: 'Afghan hound, Afghan',
 161: 'basset, basset hound',
 162: 'beagle',
 163: 'bloodhound, sleuthhound',
 164: 'bluetick',
 165: 'black-and-tan coonhound',
 166: 'Walker hound, Walker foxhound',
 167: 'English foxhound',
 168: 'redbone (dog)',
 169: 'borzoi, Russian wolfhound',
 170: 'Irish wolfhound',
 171: 'Italian greyhound',
 172: 'whippet',
 173: 'Ibizan hound, Ibizan Podenco',
 174: 'Norwegian elkhound, elkhound',
 175: 'otterhound, otter hound',
 176: 'Saluki, gazelle hound',
 177: 'Scottish deerhound, deerhound',
 178: 'Weimaraner',
 179: 'Staffordshire bullterrier, Staffordshire bull terrier',
 180: 'American Staffordshire terrier, Staffordshire terrier, American pit bull terrier, pit bull terrier',
 181: 'Bedlington terrier',
 182: 'Border terrier',
 183: 'Kerry blue terrier',
 184: 'Irish terrier',
 185: 'Norfolk terrier',
 186: 'Norwich terrier',
 187: 'Yorkshire terrier',
 188: 'wire-haired fox terrier',
 189: 'Lakeland terrier',
 190: 'Sealyham terrier, Sealyham',
 191: 'Airedale, Airedale terrier',
 192: 'cairn, cairn terrier',
 193: 'Australian terrier',
 194: 'Dandie Dinmont, Dandie Dinmont terrier',
 195: 'Boston bull, Boston terrier',
 196: 'miniature schnauzer',
 197: 'giant schnauzer',
 198: 'standard schnauzer',
 199: 'Scotch terrier, Scottish terrier, Scottie',
 200: 'Tibetan terrier, chrysanthemum dog',
 201: 'silky terrier, Sydney silky',
 202: 'soft-coated wheaten terrier',
 203: 'West Highland white terrier',
 204: 'Lhasa, Lhasa apso',
 205: 'flat-coated retriever',
 206: 'curly-coated retriever',
 207: 'golden retriever',
 208: 'Labrador retriever',
 209: 'Chesapeake Bay retriever',
 210: 'German short-haired pointer',
 211: 'vizsla, Hungarian pointer',
 212: 'English setter',
 213: 'Irish setter, red setter',
 214: 'Gordon setter',
 215: 'Brittany spaniel',
 216: 'clumber, clumber spaniel',
 217: 'English springer, English springer spaniel',
 218: 'Welsh springer spaniel',
 219: 'cocker spaniel, English cocker spaniel, cocker',
 220: 'Sussex spaniel',
 221: 'Irish water spaniel',
 222: 'kuvasz',
 223: 'schipperke',
 224: 'groenendael',
 225: 'malinois',
 226: 'briard',
 227: 'kelpie',
 228: 'komondor',
 229: 'Old English sheepdog, bobtail',
 230: 'Shetland sheepdog, Shetland sheep dog, Shetland',
 231: 'collie',
 232: 'Border collie',
 233: 'Bouvier des Flandres, Bouviers des Flandres',
 234: 'Rottweiler',
 235: 'German shepherd, German shepherd dog, German police dog, alsatian',
 236: 'Doberman, Doberman pinscher',
 237: 'miniature pinscher',
 238: 'Greater Swiss Mountain dog',
 239: 'Bernese mountain dog',
 240: 'Appenzeller',
 241: 'EntleBucher',
 242: 'boxer',
 243: 'bull mastiff',
 244: 'Tibetan mastiff',
 245: 'French bulldog',
 246: 'Great Dane',
 247: 'Saint Bernard, St Bernard',
 248: 'Eskimo dog, husky',
 249: 'malamute, malemute, Alaskan malamute',
 250: 'Siberian husky',
 251: 'dalmatian, coach dog, carriage dog',
 252: 'affenpinscher, monkey pinscher, monkey dog',
 253: 'basenji',
 254: 'pug, pug-dog',
 255: 'Leonberg',
 256: 'Newfoundland, Newfoundland dog',
 257: 'Great Pyrenees',
 258: 'Samoyed, Samoyede',
 259: 'Pomeranian',
 260: 'chow, chow chow',
 261: 'keeshond',
 262: 'Brabancon griffon',
 263: 'Pembroke, Pembroke Welsh corgi',
 264: 'Cardigan, Cardigan Welsh corgi',
 265: 'toy poodle',
 266: 'miniature poodle',
 267: 'standard poodle',
 268: 'Mexican hairless',
 269: 'timber wolf, grey wolf, gray wolf, Canis lupus',
 270: 'white wolf, Arctic wolf, Canis lupus tundrarum',
 271: 'red wolf, maned wolf, Canis rufus, Canis niger',
 272: 'coyote, prairie wolf, brush wolf, Canis latrans',
 273: 'dingo, warrigal, warragal, Canis dingo',
 274: 'dhole, Cuon alpinus',
 275: 'African hunting dog, hyena dog, Cape hunting dog, Lycaon pictus',
 276: 'hyena, hyaena',
 277: 'red fox, Vulpes vulpes',
 278: 'kit fox, Vulpes macrotis',
 279: 'Arctic fox, white fox, Alopex lagopus',
 280: 'grey fox, gray fox, Urocyon cinereoargenteus',
 281: 'tabby, tabby cat',
 282: 'tiger cat',
 283: 'Persian cat',
 284: 'Siamese cat, Siamese',
 285: 'Egyptian cat',
 286: 'cougar, puma, catamount, mountain lion, painter, panther, Felis concolor',
 287: 'lynx, catamount',
 288: 'leopard, Panthera pardus',
 289: 'snow leopard, ounce, Panthera uncia',
 290: 'jaguar, panther, Panthera onca, Felis onca',
 291: 'lion, king of beasts, Panthera leo',
 292: 'tiger, Panthera tigris',
 293: 'cheetah, chetah, Acinonyx jubatus',
 294: 'brown bear, bruin, Ursus arctos',
 295: 'American black bear, black bear, Ursus americanus, Euarctos americanus',
 296: 'ice bear, polar bear, Ursus Maritimus, Thalarctos maritimus',
 297: 'sloth bear, Melursus ursinus, Ursus ursinus',
 298: 'mongoose',
 299: 'meerkat, mierkat',
 300: 'tiger beetle',
 301: 'ladybug, ladybeetle, lady beetle, ladybird, ladybird beetle',
 302: 'ground beetle, carabid beetle',
 303: 'long-horned beetle, longicorn, longicorn beetle',
 304: 'leaf beetle, chrysomelid',
 305: 'dung beetle',
 306: 'rhinoceros beetle',
 307: 'weevil',
 308: 'fly',
 309: 'bee',
 310: 'ant, emmet, pismire',
 311: 'grasshopper, hopper',
 312: 'cricket',
 313: 'walking stick, walkingstick, stick insect',
 314: 'cockroach, roach',
 315: 'mantis, mantid',
 316: 'cicada, cicala',
 317: 'leafhopper',
 318: 'lacewing, lacewing fly',
 319: "dragonfly, darning needle, devil's darning needle, sewing needle, snake feeder, snake doctor, mosquito hawk, skeeter hawk",
 320: 'damselfly',
 321: 'admiral',
 322: 'ringlet, ringlet butterfly',
 323: 'monarch, monarch butterfly, milkweed butterfly, Danaus plexippus',
 324: 'cabbage butterfly',
 325: 'sulphur butterfly, sulfur butterfly',
 326: 'lycaenid, lycaenid butterfly',
 327: 'starfish, sea star',
 328: 'sea urchin',
 329: 'sea cucumber, holothurian',
 330: 'wood rabbit, cottontail, cottontail rabbit',
 331: 'hare',
 332: 'Angora, Angora rabbit',
 333: 'hamster',
 334: 'porcupine, hedgehog',
 335: 'fox squirrel, eastern fox squirrel, Sciurus niger',
 336: 'marmot',
 337: 'beaver',
 338: 'guinea pig, Cavia cobaya',
 339: 'sorrel horse',
 340: 'zebra',
 341: 'hog, pig, grunter, squealer, Sus scrofa',
 342: 'wild boar, boar, Sus scrofa',
 343: 'warthog',
 344: 'hippopotamus, hippo, river horse, Hippopotamus amphibius',
 345: 'ox',
 346: 'water buffalo, water ox, Asiatic buffalo, Bubalus bubalis',
 347: 'bison',
 348: 'ram, tup',
 349: 'bighorn, bighorn sheep, cimarron, Rocky Mountain bighorn, Rocky Mountain sheep, Ovis canadensis',
 350: 'ibex, Capra ibex',
 351: 'hartebeest',
 352: 'impala, Aepyceros melampus',
 353: 'gazelle',
 354: 'Arabian camel, dromedary, Camelus dromedarius',
 355: 'llama',
 356: 'weasel',
 357: 'mink',
 358: 'polecat, fitch, foulmart, foumart, Mustela putorius',
 359: 'black-footed ferret, ferret, Mustela nigripes',
 360: 'otter',
 361: 'skunk, polecat, wood pussy',
 362: 'badger',
 363: 'armadillo',
 364: 'three-toed sloth, ai, Bradypus tridactylus',
 365: 'orangutan, orang, orangutang, Pongo pygmaeus',
 366: 'gorilla, Gorilla gorilla',
 367: 'chimpanzee, chimp, Pan troglodytes',
 368: 'gibbon, Hylobates lar',
 369: 'siamang, Hylobates syndactylus, Symphalangus syndactylus',
 370: 'guenon, guenon monkey',
 371: 'patas, hussar monkey, Erythrocebus patas',
 372: 'baboon',
 373: 'macaque',
 374: 'langur',
 375: 'colobus, colobus monkey',
 376: 'proboscis monkey, Nasalis larvatus',
 377: 'marmoset',
 378: 'capuchin, ringtail, Cebus capucinus',
 379: 'howler monkey, howler',
 380: 'titi, titi monkey',
 381: 'spider monkey, Ateles geoffroyi',
 382: 'squirrel monkey, Saimiri sciureus',
 383: 'Madagascar cat, ring-tailed lemur, Lemur catta',
 384: 'indri, indris, Indri indri, Indri brevicaudatus',
 385: 'Indian elephant, Elephas maximus',
 386: 'African elephant, Loxodonta africana',
 387: 'lesser panda, red panda, panda, bear cat, cat bear, Ailurus fulgens',
 388: 'giant panda, panda, panda bear, coon bear, Ailuropoda melanoleuca',
 389: 'barracouta, snoek',
 390: 'eel',
 391: 'coho, cohoe, coho salmon, blue jack, silver salmon, Oncorhynchus kisutch',
 392: 'rock beauty, Holocanthus tricolor',
 393: 'anemone fish',
 394: 'sturgeon',
 395: 'gar, garfish, garpike, billfish, Lepisosteus osseus',
 396: 'lionfish',
 397: 'puffer, pufferfish, blowfish, globefish',
 398: 'abacus',
 399: 'abaya',
 400: "academic gown, academic robe, judge's robe",
 401: 'accordion, piano accordion, squeeze box',
 402: 'acoustic guitar',
 403: 'aircraft carrier, carrier, flattop, attack aircraft carrier',
 404: 'airliner',
 405: 'airship, dirigible',
 406: 'altar',
 407: 'ambulance',
 408: 'amphibian, amphibious vehicle',
 409: 'analog clock',
 410: 'apiary, bee house',
 411: 'apron',
 412: 'ashcan, trash can, garbage can, wastebin, ash bin, ash-bin, ashbin, dustbin, trash barrel, trash bin',
 413: 'assault rifle, assault gun',
 414: 'backpack, back pack, knapsack, packsack, rucksack, haversack',
 415: 'bakery, bakeshop, bakehouse',
 416: 'balance beam, beam',
 417: 'balloon',
 418: 'ballpoint, ballpoint pen, ballpen, Biro',
 419: 'Band Aid',
 420: 'banjo',
 421: 'bannister, banister, balustrade, balusters, handrail',
 422: 'barbell',
 423: 'barber chair',
 424: 'barbershop',
 425: 'barn',
 426: 'barometer',
 427: 'barrel, cask',
 428: 'barrow, garden cart, lawn cart, wheelbarrow',
 429: 'baseball',
 430: 'basketball',
 431: 'bassinet',
 432: 'bassoon',
 433: 'bathing cap, swimming cap',
 434: 'bath towel',
 435: 'bathtub, bathing tub, bath, tub',
 436: 'beach wagon, station wagon, wagon, estate car, beach waggon, station waggon, waggon',
 437: 'beacon, lighthouse, beacon light, pharos',
 438: 'beaker',
 439: 'bearskin, busby, shako',
 440: 'beer bottle',
 441: 'beer glass',
 442: 'bell cote, bell cot',
 443: 'bib',
 444: 'bicycle-built-for-two, tandem bicycle, tandem',
 445: 'bikini, two-piece',
 446: 'binder, ring-binder',
 447: 'binoculars, field glasses, opera glasses',
 448: 'birdhouse',
 449: 'boathouse',
 450: 'bobsled, bobsleigh, bob',
 451: 'bolo tie, bolo, bola tie, bola',
 452: 'bonnet, poke bonnet',
 453: 'bookcase',
 454: 'bookshop, bookstore, bookstall',
 455: 'bottlecap',
 456: 'bow',
 457: 'bow tie, bow-tie, bowtie',
 458: 'brass, memorial tablet, plaque',
 459: 'brassiere, bra, bandeau',
 460: 'breakwater, groin, groyne, mole, bulwark, seawall, jetty',
 461: 'breastplate, aegis, egis',
 462: 'broom',
 463: 'bucket, pail',
 464: 'buckle',
 465: 'bulletproof vest',
 466: 'bullet train, bullet',
 467: 'butcher shop, meat market',
 468: 'cab, hack, taxi, taxicab',
 469: 'caldron, cauldron',
 470: 'candle, taper, wax light',
 471: 'cannon',
 472: 'canoe',
 473: 'can opener, tin opener',
 474: 'cardigan',
 475: 'car mirror',
 476: 'carousel, carrousel, merry-go-round, roundabout, whirligig',
 477: "carpenter's kit, tool kit",
 478: 'carton',
 479: 'car wheel',
 480: 'cash machine, cash dispenser, automated teller machine, automatic teller machine, automated teller, automatic teller, ATM',
 481: 'cassette',
 482: 'cassette player',
 483: 'castle',
 484: 'catamaran',
 485: 'CD player',
 486: 'cello, violoncello',
 487: 'cellular telephone, cellular phone, cellphone, cell, mobile phone',
 488: 'chain',
 489: 'chainlink fence',
 490: 'chain mail, ring mail, mail, chain armor, chain armour, ring armor, ring armour',
 491: 'chain saw, chainsaw',
 492: 'chest',
 493: 'chiffonier, commode',
 494: 'chime, bell, gong',
 495: 'china cabinet, china closet',
 496: 'Christmas stocking',
 497: 'church, church building',
 498: 'cinema, movie theater, movie theatre, movie house, picture palace',
 499: 'cleaver, meat cleaver, chopper',
 500: 'cliff dwelling',
 501: 'cloak',
 502: 'clog, geta, patten, sabot',
 503: 'cocktail shaker',
 504: 'coffee mug',
 505: 'coffeepot',
 506: 'coil, spiral, volute, whorl, helix',
 507: 'combination lock',
 508: 'computer keyboard, keypad',
 509: 'confectionery, confectionary, candy store',
 510: 'container ship, containership, container vessel',
 511: 'convertible',
 512: 'corkscrew, bottle screw',
 513: 'cornet, horn, trumpet, trump',
 514: 'cowboy boot',
 515: 'cowboy hat, ten-gallon hat',
 516: 'cradle',
 517: 'crane',
 518: 'crash helmet',
 519: 'crate',
 520: 'crib, cot',
 521: 'Crock Pot',
 522: 'croquet ball',
 523: 'crutch',
 524: 'cuirass',
 525: 'dam, dike, dyke',
 526: 'desk',
 527: 'desktop computer',
 528: 'dial telephone, dial phone',
 529: 'diaper, nappy, napkin',
 530: 'digital clock',
 531: 'digital watch',
 532: 'dining table, board',
 533: 'dishrag, dishcloth',
 534: 'dishwasher, dish washer, dishwashing machine',
 535: 'disk brake, disc brake',
 536: 'dock, dockage, docking facility',
 537: 'dogsled, dog sled, dog sleigh',
 538: 'dome',
 539: 'doormat, welcome mat',
 540: 'drilling platform, offshore rig',
 541: 'drum, membranophone, tympan',
 542: 'drumstick',
 543: 'dumbbell',
 544: 'Dutch oven',
 545: 'electric fan, blower',
 546: 'electric guitar',
 547: 'electric locomotive',
 548: 'entertainment center',
 549: 'envelope',
 550: 'espresso maker',
 551: 'face powder',
 552: 'feather boa, boa',
 553: 'file, file cabinet, filing cabinet',
 554: 'fireboat',
 555: 'fire engine, fire truck',
 556: 'fire screen, fireguard',
 557: 'flagpole, flagstaff',
 558: 'flute, transverse flute',
 559: 'folding chair',
 560: 'football helmet',
 561: 'forklift',
 562: 'fountain',
 563: 'fountain pen',
 564: 'four-poster',
 565: 'freight car',
 566: 'French horn, horn',
 567: 'frying pan, frypan, skillet',
 568: 'fur coat',
 569: 'garbage truck, dustcart',
 570: 'gasmask, respirator, gas helmet',
 571: 'gas pump, gasoline pump, petrol pump, island dispenser',
 572: 'goblet',
 573: 'go-kart',
 574: 'golf ball',
 575: 'golfcart, golf cart',
 576: 'gondola',
 577: 'gong, tam-tam',
 578: 'gown',
 579: 'grand piano, grand',
 580: 'greenhouse, nursery, glasshouse',
 581: 'grille, radiator grille',
 582: 'grocery store, grocery, food market, market',
 583: 'guillotine',
 584: 'hair slide',
 585: 'hair spray',
 586: 'half track',
 587: 'hammer',
 588: 'hamper',
 589: 'hand blower, blow dryer, blow drier, hair dryer, hair drier',
 590: 'hand-held computer, hand-held microcomputer',
 591: 'handkerchief, hankie, hanky, hankey',
 592: 'hard disc, hard disk, fixed disk',
 593: 'harmonica, mouth organ, harp, mouth harp',
 594: 'harp',
 595: 'harvester, reaper',
 596: 'hatchet',
 597: 'holster',
 598: 'home theater, home theatre',
 599: 'honeycomb',
 600: 'hook, claw',
 601: 'hoopskirt, crinoline',
 602: 'horizontal bar, high bar',
 603: 'horse cart, horse-cart',
 604: 'hourglass',
 605: 'iPod',
 606: 'iron, smoothing iron',
 607: "jack-o'-lantern",
 608: 'jean, blue jean, denim',
 609: 'jeep, landrover',
 610: 'jersey, T-shirt, tee shirt',
 611: 'jigsaw puzzle',
 612: 'jinrikisha, ricksha, rickshaw',
 613: 'joystick',
 614: 'kimono',
 615: 'knee pad',
 616: 'knot',
 617: 'lab coat, laboratory coat',
 618: 'ladle',
 619: 'lampshade, lamp shade',
 620: 'laptop, laptop computer',
 621: 'lawn mower, mower',
 622: 'lens cap, lens cover',
 623: 'letter opener, paper knife, paperknife',
 624: 'library',
 625: 'lifeboat',
 626: 'lighter, light, igniter, ignitor',
 627: 'limousine, limo',
 628: 'liner, ocean liner',
 629: 'lipstick, lip rouge',
 630: 'Loafer',
 631: 'lotion',
 632: 'loudspeaker, speaker, speaker unit, loudspeaker system, speaker system',
 633: "loupe, jeweler's loupe",
 634: 'lumbermill, sawmill',
 635: 'magnetic compass',
 636: 'mailbag, postbag',
 637: 'mailbox, letter box',
 638: 'maillot',
 639: 'maillot, tank suit',
 640: 'manhole cover',
 641: 'maraca',
 642: 'marimba, xylophone',
 643: 'mask',
 644: 'matchstick',
 645: 'maypole',
 646: 'maze, labyrinth',
 647: 'measuring cup',
 648: 'medicine chest, medicine cabinet',
 649: 'megalith, megalithic structure',
 650: 'microphone, mike',
 651: 'microwave, microwave oven',
 652: 'military uniform',
 653: 'milk can',
 654: 'minibus',
 655: 'miniskirt, mini',
 656: 'minivan',
 657: 'missile',
 658: 'mitten',
 659: 'mixing bowl',
 660: 'mobile home, manufactured home',
 661: 'Model T',
 662: 'modem',
 663: 'monastery',
 664: 'monitor',
 665: 'moped',
 666: 'mortar',
 667: 'mortarboard',
 668: 'mosque',
 669: 'mosquito net',
 670: 'motor scooter, scooter',
 671: 'mountain bike, all-terrain bike, off-roader',
 672: 'mountain tent',
 673: 'mouse, computer mouse',
 674: 'mousetrap',
 675: 'moving van',
 676: 'muzzle',
 677: 'nail',
 678: 'neck brace',
 679: 'necklace',
 680: 'nipple',
 681: 'notebook, notebook computer',
 682: 'obelisk',
 683: 'oboe, hautboy, hautbois',
 684: 'ocarina, sweet potato',
 685: 'odometer, hodometer, mileometer, milometer',
 686: 'oil filter',
 687: 'organ, pipe organ',
 688: 'oscilloscope, scope, cathode-ray oscilloscope, CRO',
 689: 'overskirt',
 690: 'oxcart',
 691: 'oxygen mask',
 692: 'packet',
 693: 'paddle, boat paddle',
 694: 'paddlewheel, paddle wheel',
 695: 'padlock',
 696: 'paintbrush',
 697: "pajama, pyjama, pj's, jammies",
 698: 'palace',
 699: 'panpipe, pandean pipe, syrinx',
 700: 'paper towel',
 701: 'parachute, chute',
 702: 'parallel bars, bars',
 703: 'park bench',
 704: 'parking meter',
 705: 'passenger car, coach, carriage',
 706: 'patio, terrace',
 707: 'pay-phone, pay-station',
 708: 'pedestal, plinth, footstall',
 709: 'pencil box, pencil case',
 710: 'pencil sharpener',
 711: 'perfume, essence',
 712: 'Petri dish',
 713: 'photocopier',
 714: 'pick, plectrum, plectron',
 715: 'pickelhaube',
 716: 'picket fence, paling',
 717: 'pickup, pickup truck',
 718: 'pier',
 719: 'piggy bank, penny bank',
 720: 'pill bottle',
 721: 'pillow',
 722: 'ping-pong ball',
 723: 'pinwheel',
 724: 'pirate, pirate ship',
 725: 'pitcher, ewer',
 726: "plane, carpenter's plane, woodworking plane",
 727: 'planetarium',
 728: 'plastic bag',
 729: 'plate rack',
 730: 'plow, plough',
 731: "plunger, plumber's helper",
 732: 'Polaroid camera, Polaroid Land camera',
 733: 'pole',
 734: 'police van, police wagon, paddy wagon, patrol wagon, wagon, black Maria',
 735: 'poncho',
 736: 'pool table, billiard table, snooker table',
 737: 'pop bottle, soda bottle',
 738: 'pot, flowerpot',
 739: "potter's wheel",
 740: 'power drill',
 741: 'prayer rug, prayer mat',
 742: 'printer',
 743: 'prison, prison house',
 744: 'projectile, missile',
 745: 'projector',
 746: 'puck, hockey puck',
 747: 'punching bag, punch bag, punching ball, punchball',
 748: 'purse',
 749: 'quill, quill pen',
 750: 'quilt, comforter, comfort, puff',
 751: 'racer, race car, racing car',
 752: 'racket, racquet',
 753: 'radiator',
 754: 'radio, wireless',
 755: 'radio telescope, radio reflector',
 756: 'rain barrel',
 757: 'recreational vehicle, RV, R.V.',
 758: 'reel',
 759: 'reflex camera',
 760: 'refrigerator, icebox',
 761: 'remote control, remote',
 762: 'restaurant, eating house, eating place, eatery',
 763: 'revolver, six-gun, six-shooter',
 764: 'rifle',
 765: 'rocking chair, rocker',
 766: 'rotisserie',
 767: 'rubber eraser, rubber, pencil eraser',
 768: 'rugby ball',
 769: 'rule, ruler',
 770: 'running shoe',
 771: 'safe',
 772: 'safety pin',
 773: 'saltshaker, salt shaker',
 774: 'sandal',
 775: 'sarong',
 776: 'sax, saxophone',
 777: 'scabbard',
 778: 'scale, weighing machine',
 779: 'school bus',
 780: 'schooner',
 781: 'scoreboard',
 782: 'screen, CRT screen',
 783: 'screw',
 784: 'screwdriver',
 785: 'seat belt, seatbelt',
 786: 'sewing machine',
 787: 'shield, buckler',
 788: 'shoe shop, shoe-shop, shoe store',
 789: 'shoji',
 790: 'shopping basket',
 791: 'shopping cart',
 792: 'shovel',
 793: 'shower cap',
 794: 'shower curtain',
 795: 'ski',
 796: 'ski mask',
 797: 'sleeping bag',
 798: 'slide rule, slipstick',
 799: 'sliding door',
 800: 'slot, one-armed bandit',
 801: 'snorkel',
 802: 'snowmobile',
 803: 'snowplow, snowplough',
 804: 'soap dispenser',
 805: 'soccer ball',
 806: 'sock',
 807: 'solar dish, solar collector, solar furnace',
 808: 'sombrero',
 809: 'soup bowl',
 810: 'space bar',
 811: 'space heater',
 812: 'space shuttle',
 813: 'spatula',
 814: 'speedboat',
 815: "spider web, spider's web",
 816: 'spindle',
 817: 'sports car, sport car',
 818: 'spotlight, spot',
 819: 'stage',
 820: 'steam locomotive',
 821: 'steel arch bridge',
 822: 'steel drum',
 823: 'stethoscope',
 824: 'stole',
 825: 'stone wall',
 826: 'stopwatch, stop watch',
 827: 'stove',
 828: 'strainer',
 829: 'streetcar, tram, tramcar, trolley, trolley car',
 830: 'stretcher',
 831: 'studio couch, day bed',
 832: 'stupa, tope',
 833: 'submarine, pigboat, sub, U-boat',
 834: 'suit, suit of clothes',
 835: 'sundial',
 836: 'sunglass',
 837: 'sunglasses, dark glasses, shades',
 838: 'sunscreen, sunblock, sun blocker',
 839: 'suspension bridge',
 840: 'swab, swob, mop',
 841: 'sweatshirt',
 842: 'swimming trunks, bathing trunks',
 843: 'swing',
 844: 'switch, electric switch, electrical switch',
 845: 'syringe',
 846: 'table lamp',
 847: 'tank, army tank, armored combat vehicle, armoured combat vehicle',
 848: 'tape player',
 849: 'teapot',
 850: 'teddy, teddy bear',
 851: 'television, television system',
 852: 'tennis ball',
 853: 'thatch, thatched roof',
 854: 'theater curtain, theatre curtain',
 855: 'thimble',
 856: 'thresher, thrasher, threshing machine',
 857: 'throne',
 858: 'tile roof',
 859: 'toaster',
 860: 'tobacco shop, tobacconist shop, tobacconist',
 861: 'toilet seat',
 862: 'torch',
 863: 'totem pole',
 864: 'tow truck, tow car, wrecker',
 865: 'toyshop',
 866: 'tractor',
 867: 'trailer truck, tractor trailer, trucking rig, rig, articulated lorry, semi',
 868: 'tray',
 869: 'trench coat',
 870: 'tricycle, trike, velocipede',
 871: 'trimaran',
 872: 'tripod',
 873: 'triumphal arch',
 874: 'trolleybus, trolley coach, trackless trolley',
 875: 'trombone',
 876: 'tub, vat',
 877: 'turnstile',
 878: 'typewriter keyboard',
 879: 'umbrella',
 880: 'unicycle, monocycle',
 881: 'upright, upright piano',
 882: 'vacuum, vacuum cleaner',
 883: 'vase',
 884: 'vault',
 885: 'velvet',
 886: 'vending machine',
 887: 'vestment',
 888: 'viaduct',
 889: 'violin, fiddle',
 890: 'volleyball',
 891: 'waffle iron',
 892: 'wall clock',
 893: 'wallet, billfold, notecase, pocketbook',
 894: 'wardrobe, closet, press',
 895: 'warplane, military plane',
 896: 'washbasin, handbasin, washbowl, lavabo, wash-hand basin',
 897: 'washer, automatic washer, washing machine',
 898: 'water bottle',
 899: 'water jug',
 900: 'water tower',
 901: 'whiskey jug',
 902: 'whistle',
 903: 'wig',
 904: 'window screen',
 905: 'window shade',
 906: 'Windsor tie',
 907: 'wine bottle',
 908: 'wing',
 909: 'wok',
 910: 'wooden spoon',
 911: 'wool, woolen, woollen',
 912: 'worm fence, snake fence, snake-rail fence, Virginia fence',
 913: 'wreck',
 914: 'yawl',
 915: 'yurt',
 916: 'web site, website, internet site, site',
 917: 'comic book',
 918: 'crossword puzzle, crossword',
 919: 'street sign',
 920: 'traffic light, traffic signal, stoplight',
 921: 'book jacket, dust cover, dust jacket, dust wrapper',
 922: 'menu',
 923: 'plate',
 924: 'guacamole',
 925: 'consomme',
 926: 'hot pot, hotpot',
 927: 'trifle',
 928: 'ice cream, icecream',
 929: 'ice lolly, lolly, lollipop, popsicle',
 930: 'French loaf',
 931: 'bagel, beigel',
 932: 'pretzel',
 933: 'cheeseburger',
 934: 'hotdog, hot dog, red hot',
 935: 'mashed potato',
 936: 'head cabbage',
 937: 'broccoli',
 938: 'cauliflower',
 939: 'zucchini, courgette',
 940: 'spaghetti squash',
 941: 'acorn squash',
 942: 'butternut squash',
 943: 'cucumber, cuke',
 944: 'artichoke, globe artichoke',
 945: 'bell pepper',
 946: 'cardoon',
 947: 'mushroom',
 948: 'Granny Smith',
 949: 'strawberry',
 950: 'orange',
 951: 'lemon',
 952: 'fig',
 953: 'pineapple, ananas',
 954: 'banana',
 955: 'jackfruit, jak, jack',
 956: 'custard apple',
 957: 'pomegranate',
 958: 'hay',
 959: 'carbonara',
 960: 'chocolate sauce, chocolate syrup',
 961: 'dough',
 962: 'meat loaf, meatloaf',
 963: 'pizza, pizza pie',
 964: 'potpie',
 965: 'burrito',
 966: 'red wine',
 967: 'espresso',
 968: 'cup',
 969: 'eggnog',
 970: 'alp',
 971: 'bubble',
 972: 'cliff, drop, drop-off',
 973: 'coral reef',
 974: 'geyser',
 975: 'lakeside, lakeshore',
 976: 'promontory, headland, head, foreland',
 977: 'sandbar, sand bar',
 978: 'seashore, coast, seacoast, sea-coast',
 979: 'valley, vale',
 980: 'volcano',
 981: 'ballplayer, baseball player',
 982: 'groom, bridegroom',
 983: 'scuba diver',
 984: 'rapeseed',
 985: 'daisy',
 986: "yellow lady's slipper, yellow lady-slipper, Cypripedium calceolus, Cypripedium parviflorum",
 987: 'corn',
 988: 'acorn',
 989: 'hip, rose hip, rosehip',
 990: 'buckeye, horse chestnut, conker',
 991: 'coral fungus',
 992: 'agaric',
 993: 'gyromitra',
 994: 'stinkhorn, carrion fungus',
 995: 'earthstar',
 996: 'hen-of-the-woods, hen of the woods, Polyporus frondosus, Grifola frondosa',
 997: 'bolete',
 998: 'ear, spike, capitulum',
 999: 'toilet tissue, toilet paper, bathroom tissue'}

In [ ]:
di_large = WeightedCoordinateDescent(
    rounds=1,samples_per_dim=[30,])
di_large = WCD_LATTICE_WRAPPER(di_large)

In [ ]:

# results = visualize_rotation_optimization(
#     model=model,
#     optimizer=di_large,
#     problem=problem_nn_pytorch_pretrained,
#     test_loader=test_loader,
#     class_names=class_map,
#     confidence_module=problem_nn_pytorch_pretrained.confidence_module,
#     num_rotations=8,
#     sample_idx=14,
#     device=device,
#     save_dir=savepath("visualizations/sample_14")
# )


In [ ]:
problem_energy_logits = TransformationProblem(energy_confidence, transform_seq,
                                                      consolidate_method="consolidate_simple",max_batch_size=32)

load_or_run_evaluate_confidence_and_search(
    model, optimizer=di, problem=problem_energy_logits,
    test_loader=test_loader_transformed, max_batch_override=dataset_info.batch_size_search,
    save_path=savepath("final_results_energy"),show_progress=True,
    repeats=1)

In [ ]:
load_or_run_evaluate_confidence_and_search(
    model, optimizer=di, problem=problem_energy_logits,
    test_loader=test_loader_transformed, max_batch_override=dataset_info.batch_size_search,
    save_path=savepath("final_results_energy"),show_progress=True,
    repeats=2)

In [ ]:
load_or_run_evaluate_confidence_and_search(
    model, optimizer=di, problem=problem_energy_logits,
    test_loader=test_loader_transformed, max_batch_override=dataset_info.batch_size_search,
    save_path=savepath("final_results_energy"),show_progress=True,
    repeats=3)

In [ ]:
load_or_run_evaluate_confidence_and_search(
    model, optimizer=di, problem=problem_energy_logits,
    test_loader=test_loader_transformed, max_batch_override=dataset_info.batch_size_search,
    save_path=savepath("final_results_energy"),show_progress=True,
    repeats=4)

In [ ]:
import torch
import matplotlib.pyplot as plt
import torchvision
from matplotlib.backends.backend_pdf import PdfPages

# --- Unnormalize helper ---
def unnormalize(img, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]):
    mean = torch.tensor(mean,device=img.device).view(3, 1, 1)
    std = torch.tensor(std,device=img.device).view(3, 1, 1)
    return img * std + mean


def show_and_save_batch(loader, title, nrow=4, pdf_prefix="output"):
    imgs, _ = next(iter(loader))

    # Unnormalize
    imgs_unnorm = unnormalize(imgs)

    # Ensure enough images for a good grid
    if len(imgs_unnorm) < nrow * nrow:
        imgs2, _ = next(iter(loader))
        imgs_unnorm = torch.cat([imgs_unnorm, unnormalize(imgs2)], dim=0)

    # Build image grid
    grid = torchvision.utils.make_grid(
        imgs_unnorm[:nrow * nrow], nrow=nrow
    ).permute(1, 2, 0).cpu().clip(0, 1)

    # Show
    plt.figure(figsize=(10, 10))
    plt.imshow(grid)
    plt.axis("off")
    plt.show()

    # Save grid as ONE PDF, using your savepath()
    pdf_file = savepath(f"{pdf_prefix}_{title}.pdf")
    #remove json ending
    pdf_file = pdf_file.replace(".json", "")
    with PdfPages(pdf_file) as pdf:
        fig = plt.figure(figsize=(10, 10))
        plt.imshow(grid)
        plt.axis("off")
        pdf.savefig(fig)
        plt.close(fig)

    print(f"Saved PDF grid → {pdf_file}")





In [ ]:
W = plt_setup_latex()

In [ ]:
import torch
import numpy as np
import tqdm
from numpy.testing import assert_array_equal, assert_allclose

def test_loader_determinism(loader, num_batches_to_test: int = 5):
    """Tests if iterating a PyTorch DataLoader twice yields identical results."""

    print(f"Testing loader determinism over the first {num_batches_to_test} batches...")

    # --- Run 1: Collect first N batches ---
    run1_data, run1_labels = [], []
    loader_iter1 = iter(loader)
    for _ in tqdm.tqdm(range(num_batches_to_test), desc="Run 1"):
        try:
            x, y = next(loader_iter1)
            run1_data.append(x)
            run1_labels.append(y)
        except StopIteration:
            break

    # --- Run 2: Collect first N batches ---
    run2_data, run2_labels = [], []
    loader_iter2 = iter(loader)
    for _ in tqdm.tqdm(range(num_batches_to_test), desc="Run 2"):
        try:
            x, y = next(loader_iter2)
            run2_data.append(x)
            run2_labels.append(y)
        except StopIteration:
            break

    # --- Comparison ---
    if len(run1_data) != len(run2_data):
        print(f"❌ ERROR: Run 1 yielded {len(run1_data)} batches, Run 2 yielded {len(run2_data)} batches.")
        return False

    for i in range(len(run1_data)):
        # 1. Check Labels (must be exact)
        try:
            assert_array_equal(run1_labels[i].numpy(), run2_labels[i].numpy())
        except AssertionError as e:
            print(f"❌ FAILED on Batch {i}: Labels are not identical.")
            print(e)
            return False

        # 2. Check Data (allowing for float precision, though it should be exact)
        try:
            # Use allclose with a small tolerance for floating point numbers
            # If the data includes non-deterministic noise/augmentations, this will fail.
            assert torch.allclose(run1_data[i], run2_data[i], atol=1e-6)
        except AssertionError as e:
            print(f"❌ FAILED on Batch {i}: Data tensors are not identical (max diff: {(run1_data[i] - run2_data[i]).abs().max()}).")
            print("This indicates randomness in the data loading or augmentation pipeline.")
            return False

    print(" SUCCESS: Data loader is deterministic for the tested batches.")
    return True

# --- USAGE ---
# test_loader_determinism(test_loader_transformed)

In [ ]:

# --- Tradeoff utilities (copy into `experiment_thesis/imagenet_resnet.ipynb`) ---
import os
import numpy as np
import torch
import tqdm
import matplotlib.pyplot as plt

# Ensure result dirs
results_dir_trade = os.path.join(current_path, "experiment_files", "results", dataset, architecture, "comparison_tradeoff")
os.makedirs(results_dir_trade, exist_ok=True)
os.makedirs(os.path.dirname(os.path.join(embedding_cache_path, dataset, architecture, transform_name, "dummy")), exist_ok=True)

def default_cache_path(label: str):
    safe = "".join(c if c.isalnum() or c in "-_." else "_" for c in label)
    return os.path.join(embedding_cache_path, dataset, architecture, transform_name, f"{safe}_cached_confidences.npz")

@torch.no_grad()
def calculate_confidences_and_cache(
    optimizer,
    best_problem,
    conf_fn,
    test_loader,
    test_loader_transformed,
    device,
    n_runs: int = 5,
    cache_path: str = "cached_confidences.npz",
    force_recompute: bool = False,
    augment_true_func=None,
):
    if os.path.exists(cache_path) and not force_recompute:
        print(f" Loading cached results from {cache_path}")
        return dict(np.load(cache_path, allow_pickle=True))

    print("=== Calculating base confidences (no optimization) ===")
    assert test_loader_determinism(test_loader, num_batches_to_test=3), "Test loader is not deterministic!"
    assert test_loader_determinism(test_loader_transformed, num_batches_to_test=3), "Transformed test loader is not deterministic!"

    def eval_conf(loader, augment_func=None):
        all_conf, correct = [], []
        for x_test, y_test in tqdm.tqdm(loader):
            x_test = x_test.to(device)
            if augment_func is not None:
                x_test = augment_func(x_test)
            conf_values, logits = conf_fn(x_test)
            if logits is None:
                logits = model(x_test)
            pred_y = torch.argmax(logits, dim=1).cpu()
            correct.append((pred_y == y_test).numpy())
            all_conf.append(conf_values.cpu().numpy())
        return np.concatenate(correct), np.concatenate(all_conf)

    correct_predicted, all_conf = eval_conf(test_loader, augment_true_func)
    correct_predicted_transformed, all_conf_transformed = eval_conf(test_loader_transformed)

    results = {
        "correct_predicted": correct_predicted,
        "all_conf": all_conf,
        "correct_predicted_transformed": correct_predicted_transformed,
        "all_conf_transformed": all_conf_transformed,
    }

    print("\n=== Running probabilistic optimizer ===")

    optimized_correct_runs, optimized_conf_runs = [], []
    optimized_correct_runs_t, optimized_conf_runs_t = [], []

    for run_idx in range(n_runs):
        print(f"\n--- Optimization run {run_idx + 1}/{n_runs} ---")

        correct_run, conf_run = [], []
        for x_test, y_test in tqdm.tqdm(test_loader, desc=f"Test run {run_idx+1}"):
            x_test = x_test.to(device)
            p, error, _ = optimizer.optimize(best_problem, x_test)
            x_optimized = best_problem.transform_sequence.transform(x_test, p)
            conf_values, logits = conf_fn(x_optimized)
            if logits is None:
                logits = model(x_optimized)
            pred_y = torch.argmax(logits, dim=1).cpu()
            correct_run.append((pred_y == y_test).numpy())
            conf_run.append(conf_values.cpu().numpy())
        optimized_correct_runs.append(np.concatenate(correct_run))
        optimized_conf_runs.append(np.concatenate(conf_run))

        correct_run_t, conf_run_t = [], []
        for x_test, y_test in tqdm.tqdm(test_loader_transformed, desc=f"Transformed run {run_idx+1}"):
            x_test = x_test.to(device)
            p, error, _ = optimizer.optimize(best_problem, x_test)
            x_optimized = best_problem.transform_sequence.transform(x_test, p)
            conf_values, logits = conf_fn(x_optimized)
            if logits is None:
                logits = model(x_optimized)
            pred_y = torch.argmax(logits, dim=1).cpu()
            correct_run_t.append((pred_y == y_test).numpy())
            conf_run_t.append(conf_values.cpu().numpy())
        optimized_correct_runs_t.append(np.concatenate(correct_run_t))
        optimized_conf_runs_t.append(np.concatenate(conf_run_t))

    results["optimized_correct_predicted"] = np.stack(optimized_correct_runs, axis=1)
    results["optimized_all_conf"] = np.stack(optimized_conf_runs, axis=1)
    results["optimized_correct_predicted_transformed"] = np.stack(optimized_correct_runs_t, axis=1)
    results["optimized_all_conf_transformed"] = np.stack(optimized_conf_runs_t, axis=1)

    np.savez_compressed(cache_path, **results)
    print(f"\nCached results saved to: {cache_path}")
    return results
from src.utils.eval.vis import plt_setup_paper
W =plt_setup_latex()
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

def plot_confidence_threshold_results(
    results: dict,
    dataset_name: str = "Dataset",
    use_only_improved: bool = True,
    improvement_metric: str = "confidence",
    relative_improvement: float = 0.0,
    ci_multiplier: float = 1.96,
    lower_quantile: float = 0.001,
    upper_quantile: float = 1.0,
    plot_ratio=True,
    save_path=None,
    ratio_name="Optimization Ratio",
    threshold_transform=lambda x: x,
):
    if improvement_metric != "confidence":
        raise ValueError("Only improvement_metric='confidence' is supported.")

    # ---- 1. SETUP BOUNDS AND DETECT DIRECTION ----
    raw_conf_t = results["all_conf_transformed"]
    raw_conf_o = results["all_conf"]

    # Calculate quantiles in original space
    ql_raw = np.quantile(raw_conf_t, lower_quantile)
    qh_raw = np.quantile(raw_conf_t, upper_quantile)

    # Check if the transform flips the order (monotonically decreasing)
    # We test with the quantiles themselves
    ql_trans = threshold_transform(ql_raw)
    qh_trans = threshold_transform(qh_raw)
    is_decreasing = ql_trans > qh_trans

    # ---- 2. TRANSFORM DATA ----
    all_conf = threshold_transform(results["all_conf"])
    all_conf_t = threshold_transform(results["all_conf_transformed"])
    opt_conf = threshold_transform(results["optimized_all_conf"])
    opt_conf_t = threshold_transform(results["optimized_all_conf_transformed"])

    corr = results["correct_predicted"]
    corr_t = results["correct_predicted_transformed"]
    opt_corr = results["optimized_correct_predicted"]
    opt_corr_t = results["optimized_correct_predicted_transformed"]

    # Transform range for relative improvement scaling
    min_conf_trans = float(min(all_conf.min(), all_conf_t.min()))
    max_conf_trans = float(max(all_conf.max(), all_conf_t.max()))
    conf_range_trans = abs(max_conf_trans - min_conf_trans)

    # We iterate through the ORIGINAL scale to ensure we pick the same
    # samples as the original graph, then map the threshold for the x-axis
    raw_threshold_steps = np.linspace(ql_raw, qh_raw, 400)
    plot_thresholds = threshold_transform(raw_threshold_steps)

    n_runs = opt_conf.shape[1]
    acc_orig_runs, acc_trans_runs = [], []
    ratio_orig_runs, ratio_trans_runs = [], []

    # ---- 3. EVALUATE (WITH DYNAMIC INEQUALITY) ----
    for r in range(n_runs):
        acc_o, acc_t = [], []
        ratio_o, ratio_t = [], []

        for i in range(len(raw_threshold_steps)):
            raw_thr = raw_threshold_steps[i]
            trans_thr = plot_thresholds[i]

            # CRITICAL: If transform is decreasing, 'conf <= raw_thr'
            # is equivalent to 'trans_conf >= trans_thr'
            if is_decreasing:
                applied_o = all_conf >= trans_thr
                applied_t = all_conf_t >= trans_thr
            else:
                applied_o = all_conf <= trans_thr
                applied_t = all_conf_t <= trans_thr

            # Comparison Logic (using transformed values)
            # We assume 'higher' in transformed space means 'better' for these checks
            # If the transform flips confidence, you might need to flip these too
            improved_o = opt_conf[:, r] >= all_conf if not is_decreasing else opt_conf[:, r] <= all_conf
            improved_t = opt_conf_t[:, r] >= all_conf_t if not is_decreasing else opt_conf_t[:, r] <= all_conf_t

            if relative_improvement > 0.0:
                rel_imp_o = abs(opt_conf[:, r] - all_conf) / conf_range_trans
                rel_imp_t = abs(opt_conf_t[:, r] - all_conf_t) / conf_range_trans
                improved_o = np.logical_and(improved_o, rel_imp_o >= relative_improvement)
                improved_t = np.logical_and(improved_t, rel_imp_t >= relative_improvement)

            attempted_o = np.logical_and(applied_o, improved_o)
            attempted_t = np.logical_and(applied_t, improved_t)

            ratio_o.append(applied_o.mean())
            ratio_t.append(applied_t.mean())

            use_opt_o = attempted_o if use_only_improved else applied_o
            use_opt_t = attempted_t if use_only_improved else applied_t

            final_corr_o = np.where(use_opt_o, opt_corr[:, r], corr)
            final_corr_t = np.where(use_opt_t, opt_corr_t[:, r], corr_t)

            acc_o.append(final_corr_o.mean())
            acc_t.append(final_corr_t.mean())

        acc_orig_runs.append(acc_o)
        acc_trans_runs.append(acc_t)
        ratio_orig_runs.append(ratio_o)
        ratio_trans_runs.append(ratio_t)

    # Convert to numpy for CI calculation
    acc_orig_runs, acc_trans_runs = np.array(acc_orig_runs), np.array(acc_trans_runs)
    ratio_orig_runs, ratio_trans_runs = np.array(ratio_orig_runs), np.array(ratio_trans_runs)

    # ---- 4. PLOTTING ----
    def compute_ci(y):
        n = y.shape[0]
        mean = y.mean(axis=0)
        se = y.std(axis=0, ddof=1) / np.sqrt(n) if n > 1 else np.zeros_like(mean)
        return mean, mean - ci_multiplier * se, mean + ci_multiplier * se

    fig, ax1 = plt.subplots(figsize=(W, W/3.1))
    tab10 = plt.get_cmap("tab10")
    col_o, col_t, col_ro, col_rt = tab10(0), tab10(1), tab10(2), tab10(3)

    ax1.grid(True, linestyle="--", alpha=0.6)

    m_o, lo_o, hi_o = compute_ci(acc_orig_runs)
    m_t, lo_t, hi_t = compute_ci(acc_trans_runs)

    ax1.plot(plot_thresholds, m_o, label="Acc. Original", color=col_o)
    ax1.fill_between(plot_thresholds, lo_o, hi_o, alpha=0.25, color=col_o)
    ax1.plot(plot_thresholds, m_t, label="Acc. Transformed", color=col_t)
    ax1.fill_between(plot_thresholds, lo_t, hi_t, alpha=0.25, color=col_t)

    ax1.set_xlabel("Threshold (Recalculated)", fontsize=9)
    ax1.set_ylabel("Accuracy", fontsize=9)

    if plot_ratio:
        ax2 = ax1.twinx()
        rm_o, rlo_o, rhi_o = compute_ci(ratio_orig_runs)
        rm_t, rlo_t, rhi_t = compute_ci(ratio_trans_runs)
        ax2.plot(plot_thresholds, rm_o, linestyle="--", color=col_ro, label="Ratio Original")
        ax2.plot(plot_thresholds, rm_t, linestyle="--", color=col_rt, label="Ratio Transformed")
        ax2.set_ylabel(ratio_name)


    ax1.set_xlim([ql_trans, qh_trans])
    if plot_ratio:
        ax2.set_xlim([ql_trans, qh_trans])
         # ---- Combined legend ----
    lines1, labels1 = ax1.get_legend_handles_labels()
    if plot_ratio:
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax2.legend(lines1 + lines2, labels1 + labels2, loc="center left", fontsize=8, framealpha=0.5,labelspacing=0.2,)
    else:
        ax1.legend(lines1, labels1, loc="center left", fontsize=8, framealpha=0.5,labelspacing=0.2,)

    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path,bbox_inches="tight")
    plt.show()

    print(f"[{dataset_name}] Direction: {'Decreasing' if is_decreasing else 'Increasing'}")
    print(f"Range: {ql_trans:.4f} to {qh_trans:.4f}")


In [ ]:
# from datasets import load_dataset
# from tqdm import tqdm
# from PIL import Image
# import os
#
# # --------------------------
# # SETTINGS (CHANGE THESE)
# # --------------------------
# SPLIT = "validation"                          # "train" / "validation" / "test"
# SAVE_DIR = "D:/imagenet_full_val"      # where to save images
# # --------------------------
#
# # Load dataset in streaming mode (best for huge datasets)
# ds = load_dataset(
#     "ILSVRC/imagenet-1k",
#     split=SPLIT,
#     streaming=True,
#     trust_remote_code=True
# )
#
# # Create base directory
# os.makedirs(SAVE_DIR, exist_ok=True)
#
# # Count tracker (optional)
# counts = {}
#
# # Progress bar
# pbar = tqdm(ds, desc=f"Saving {SPLIT}", unit="images")
#
# for i, sample in enumerate(pbar):
#     img = sample["image"]
#     label = sample["label"]
#
#     # Track per-class count
#     if label not in counts:
#         counts[label] = 0
#
#     # Create class folder
#     class_dir = os.path.join(SAVE_DIR, str(label))
#     os.makedirs(class_dir, exist_ok=True)
#
#     # File path
#     filename = os.path.join(class_dir, f"{counts[label]:06d}.jpg")
#
#     # Skip existing files → enables RESUMABLE downloads
#     if os.path.exists(filename):
#         counts[label] += 1
#         continue
#
#     # Save image
#     img = img.convert("RGB")
#     img.save(filename, "JPEG")
#
#     counts[label] += 1
#
#     # Update tqdm description
#     pbar.set_description(f"Saving {SPLIT} | Class {label}: {counts[label]} images")
#
# pbar.close()
#
# print("\n✔ Completed download of full split:", SPLIT)
# print("✔ Saved to:", SAVE_DIR)


In [ ]:
from torchvision import models
from dataset.si_score import ImageNetSubset

# Use ViT preprocessing for SI-Score dataset
vit_transform = models.ViT_B_16_Weights.IMAGENET1K_V1.transforms()




imagenet_dataset_val = ImageNetSubset(root="D:/imagenet_full_val",
                                  transform=vit_transform)

loader_in = torch.utils.data.DataLoader(imagenet_dataset_val, batch_size=32, shuffle=False, num_workers=20,
                                              pin_memory=True, persistent_workers=True,prefetch_factor=4)


In [ ]:



problem_to_use = problem_nn_pytorch_pretrained



conf_fn = problem_to_use.confidence_module


optimizer_obj = di


cache_file = default_cache_path("imagenet_tradeoff2")
results = calculate_confidences_and_cache(
    optimizer=optimizer_obj,
    best_problem=problem_to_use,
    conf_fn=conf_fn,
    test_loader=loader_in,
    test_loader_transformed=test_loader_transformed,
    device=device,
    n_runs=1,
    cache_path=cache_file,
)

plot_confidence_threshold_results(results, dataset_name=dataset, save_path=os.path.join(results_dir_trade, "confidence_threshold.pdf"),use_only_improved=True,relative_improvement=0.1)
print("Saved plot to:", os.path.join(results_dir_trade, "confidence_threshold.pdf"))

In [ ]:
import os
import numpy as np
import torch
import tqdm
import time

def seed_everything(seed: int):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Optional but recommended for reproducibility
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

@torch.no_grad()
def calculate_confidences_and_cache_resumable(
    optimizer,
    best_problem,
    conf_fn,
    test_loader,
    test_loader_transformed,
    device,
    n_runs: int = 5,
    cache_dir: str = "cached_confidences",
    force_recompute: bool = False,
    augment_true_func=None,
):
    os.makedirs(cache_dir, exist_ok=True)

    base_cache_path = os.path.join(cache_dir, "base_results.npz")

    #seed everything based on time
    seed_everything(int(time.time_ns() % (2**32 - 1)))

    # ------------------------------------------------------------------
    # Base (non-optimized) confidences
    # ------------------------------------------------------------------
    if os.path.exists(base_cache_path) and not force_recompute:
        print(f" Loading base cached results from {base_cache_path}")
        base_results = dict(np.load(base_cache_path, allow_pickle=True))
    else:
        print("=== Calculating base confidences (no optimization) ===")

        assert test_loader_determinism(test_loader, num_batches_to_test=3)
        assert test_loader_determinism(test_loader_transformed, num_batches_to_test=3)

        def eval_conf(loader, augment_func=None):
            all_conf, correct = [], []
            for x_test, y_test in tqdm.tqdm(loader):
                x_test = x_test.to(device)
                if augment_func is not None:
                    x_test = augment_func(x_test)

                conf_values, logits = conf_fn(x_test)
                if logits is None:
                    logits = model(x_test)

                pred_y = torch.argmax(logits, dim=1).cpu()
                correct.append((pred_y == y_test).numpy())
                all_conf.append(conf_values.cpu().numpy())

            return np.concatenate(correct), np.concatenate(all_conf)

        correct_predicted, all_conf = eval_conf(test_loader, augment_true_func)
        correct_predicted_t, all_conf_t = eval_conf(test_loader_transformed)

        base_results = {
            "correct_predicted": correct_predicted,
            "all_conf": all_conf,
            "correct_predicted_transformed": correct_predicted_t,
            "all_conf_transformed": all_conf_t,
        }

        np.savez_compressed(base_cache_path, **base_results)
        print(f"Saved base results to {base_cache_path}")

    # ------------------------------------------------------------------
    # Optimized runs (saved individually)
    # ------------------------------------------------------------------
    print("\n=== Running probabilistic optimizer (resumable) ===")

    for run_idx in range(n_runs):
        run_path = os.path.join(cache_dir, f"optimized_run_{run_idx}.npz")

        if os.path.exists(run_path) and not force_recompute:
            print(f"✓ Run {run_idx + 1}/{n_runs} already cached — skipping")
            continue

        print(f"\n--- Optimization run {run_idx + 1}/{n_runs} ---")

        correct_run, conf_run = [], []
        for x_test, y_test in tqdm.tqdm(test_loader, desc=f"Test run {run_idx+1}"):
            x_test = x_test.to(device)
            p, error, _ = optimizer.optimize(best_problem, x_test)
            x_opt = best_problem.transform_sequence.transform(x_test, p)

            conf_values, logits = conf_fn(x_opt)
            if logits is None:
                logits = model(x_opt)

            pred_y = torch.argmax(logits, dim=1).cpu()
            correct_run.append((pred_y == y_test).numpy())
            conf_run.append(conf_values.cpu().numpy())

        correct_run = np.concatenate(correct_run)
        conf_run = np.concatenate(conf_run)

        correct_run_t, conf_run_t = [], []
        for x_test, y_test in tqdm.tqdm(
            test_loader_transformed, desc=f"Transformed run {run_idx+1}"
        ):
            x_test = x_test.to(device)
            p, error, _ = optimizer.optimize(best_problem, x_test)
            x_opt = best_problem.transform_sequence.transform(x_test, p)

            conf_values, logits = conf_fn(x_opt)
            if logits is None:
                logits = model(x_opt)

            pred_y = torch.argmax(logits, dim=1).cpu()
            correct_run_t.append((pred_y == y_test).numpy())
            conf_run_t.append(conf_values.cpu().numpy())

        correct_run_t = np.concatenate(correct_run_t)
        conf_run_t = np.concatenate(conf_run_t)

        np.savez_compressed(
            run_path,
            optimized_correct_predicted=correct_run,
            optimized_all_conf=conf_run,
            optimized_correct_predicted_transformed=correct_run_t,
            optimized_all_conf_transformed=conf_run_t,
        )

        print(f"Saved run {run_idx + 1} to {run_path}")

    # ------------------------------------------------------------------
    # Aggregate results
    # ------------------------------------------------------------------
    optimized_correct_runs = []
    optimized_conf_runs = []
    optimized_correct_runs_t = []
    optimized_conf_runs_t = []

    for run_idx in range(n_runs):
        run_path = os.path.join(cache_dir, f"optimized_run_{run_idx}.npz")
        if not os.path.exists(run_path):
            raise RuntimeError(f"Missing cached run: {run_path}")

        run_data = np.load(run_path, allow_pickle=True)
        optimized_correct_runs.append(run_data["optimized_correct_predicted"])
        optimized_conf_runs.append(run_data["optimized_all_conf"])
        optimized_correct_runs_t.append(run_data["optimized_correct_predicted_transformed"])
        optimized_conf_runs_t.append(run_data["optimized_all_conf_transformed"])

    results = dict(base_results)
    results.update({
        "optimized_correct_predicted": np.stack(optimized_correct_runs, axis=1),
        "optimized_all_conf": np.stack(optimized_conf_runs, axis=1),
        "optimized_correct_predicted_transformed": np.stack(optimized_correct_runs_t, axis=1),
        "optimized_all_conf_transformed": np.stack(optimized_conf_runs_t, axis=1),
    })

    final_path = os.path.join(cache_dir, "final_aggregated_results.npz")
    np.savez_compressed(final_path, **results)
    print(f"\nFinal aggregated cache saved to: {final_path}")

    return results

cache_file = default_cache_path("imagenet_tradeoff2_resumable")
results = calculate_confidences_and_cache_resumable(
    optimizer=optimizer_obj,
    best_problem=problem_to_use,
    conf_fn=conf_fn,
    test_loader=loader_in,
    test_loader_transformed=test_loader_transformed,
    device=device,
    n_runs=4,
    cache_dir=cache_file,
)

plot_confidence_threshold_results(results, dataset_name=dataset, save_path=os.path.join(results_dir_trade, "confidence_threshold_resum.pdf"),use_only_improved=True,relative_improvement=0.05,ci_multiplier=40)
print("Saved plot to:", os.path.join(results_dir_trade, "confidence_threshold_resum.pdf"))

In [ ]:
@torch.no_grad()
def test_optimizer_determinism(
    optimizer,
    best_problem,
    sample_batch,
    device,
    n_trials: int = 3,
):
    x = sample_batch.to(device)

    outputs = []

    for i in range(n_trials):
        p, error, aux = optimizer.optimize(best_problem, x)

        outputs.append((
            p.detach().cpu(),
            error.detach().cpu() if torch.is_tensor(error) else error,
        ))

    # Compare all outputs to first run
    p0, e0 = outputs[0]

    for i, (pi, ei) in enumerate(outputs[1:], start=1):
        p_equal = torch.allclose(p0, pi, atol=0, rtol=0)
        e_equal = (
            torch.allclose(e0, ei, atol=0, rtol=0)
            if torch.is_tensor(ei)
            else e0 == ei
        )

        print(f"Trial {i}: p equal = {p_equal}, error equal = {e_equal}")

    return

In [ ]:
figure_path = os.path.join(current_path,"experiment_files","export","fig","comp_tradeoff",dataset,transform_name)
os.makedirs(figure_path, exist_ok=True)

In [ ]:
cache_file = default_cache_path("imagenet_tradeoff3_resumable")
results = calculate_confidences_and_cache_resumable(
    optimizer=optimizer_obj,
    best_problem=problem_to_use,
    conf_fn=conf_fn,
    test_loader=loader_in,
    test_loader_transformed=test_loader_transformed,
    device=device,
    n_runs=2,
    cache_dir=cache_file,
)
plot_confidence_threshold_results(results, dataset_name=dataset, save_path=os.path.join(figure_path, "confidence_threshold_resum2.pdf"),use_only_improved=True,relative_improvement=0.05,ci_multiplier=1.96)
print("Saved plot to:", os.path.join(results_dir_trade, "confidence_threshold_resum2.pdf"))

In [ ]:
from src.utils.eval.vis import plt_setup_paper

W = plt_setup_paper()
def plot_confidence_threshold_results(
    results: dict,
    dataset_name: str = "Dataset",
    use_only_improved: bool = True,
    improvement_metric: str = "confidence",
    relative_improvement: float = 0.0,
    ci_multiplier: float = 1.96,
    lower_quantile: float = 0.001,
    upper_quantile: float = 1.0,
    plot_ratio=True,
    save_path=None,
    ratio_name="Optimization Ratio",
    threshold_transform=lambda x: x,
):
    if improvement_metric != "confidence":
        raise ValueError("Only improvement_metric='confidence' is supported.")

    # ---- 1. SETUP BOUNDS AND DETECT DIRECTION ----
    raw_conf_t = results["all_conf_transformed"]
    raw_conf_o = results["all_conf"]

    # Calculate quantiles in original space
    ql_raw = np.quantile(raw_conf_t, lower_quantile)
    qh_raw = np.quantile(raw_conf_t, upper_quantile)

    # Check if the transform flips the order (monotonically decreasing)
    # We test with the quantiles themselves
    ql_trans = threshold_transform(ql_raw)
    qh_trans = threshold_transform(qh_raw)
    is_decreasing = ql_trans > qh_trans

    # ---- 2. TRANSFORM DATA ----
    all_conf = threshold_transform(results["all_conf"])
    all_conf_t = threshold_transform(results["all_conf_transformed"])
    opt_conf = threshold_transform(results["optimized_all_conf"])
    opt_conf_t = threshold_transform(results["optimized_all_conf_transformed"])

    corr = results["correct_predicted"]
    corr_t = results["correct_predicted_transformed"]
    opt_corr = results["optimized_correct_predicted"]
    opt_corr_t = results["optimized_correct_predicted_transformed"]

    # Transform range for relative improvement scaling
    min_conf_trans = float(min(all_conf.min(), all_conf_t.min()))
    max_conf_trans = float(max(all_conf.max(), all_conf_t.max()))
    conf_range_trans = abs(max_conf_trans - min_conf_trans)

    # We iterate through the ORIGINAL scale to ensure we pick the same
    # samples as the original graph, then map the threshold for the x-axis
    raw_threshold_steps = np.linspace(ql_raw, qh_raw, 400)
    plot_thresholds = threshold_transform(raw_threshold_steps)

    n_runs = opt_conf.shape[1]
    acc_orig_runs, acc_trans_runs = [], []
    ratio_orig_runs, ratio_trans_runs = [], []

    # ---- 3. EVALUATE (WITH DYNAMIC INEQUALITY) ----
    for r in range(n_runs):
        acc_o, acc_t = [], []
        ratio_o, ratio_t = [], []

        for i in range(len(raw_threshold_steps)):
            raw_thr = raw_threshold_steps[i]
            trans_thr = plot_thresholds[i]

            # CRITICAL: If transform is decreasing, 'conf <= raw_thr'
            # is equivalent to 'trans_conf >= trans_thr'
            if is_decreasing:
                applied_o = all_conf >= trans_thr
                applied_t = all_conf_t >= trans_thr
            else:
                applied_o = all_conf <= trans_thr
                applied_t = all_conf_t <= trans_thr

            # Comparison Logic (using transformed values)
            # We assume 'higher' in transformed space means 'better' for these checks
            # If the transform flips confidence, you might need to flip these too
            improved_o = opt_conf[:, r] >= all_conf if not is_decreasing else opt_conf[:, r] <= all_conf
            improved_t = opt_conf_t[:, r] >= all_conf_t if not is_decreasing else opt_conf_t[:, r] <= all_conf_t

            if relative_improvement > 0.0:
                rel_imp_o = abs(opt_conf[:, r] - all_conf) / conf_range_trans
                rel_imp_t = abs(opt_conf_t[:, r] - all_conf_t) / conf_range_trans
                improved_o = np.logical_and(improved_o, rel_imp_o >= relative_improvement)
                improved_t = np.logical_and(improved_t, rel_imp_t >= relative_improvement)

            attempted_o = np.logical_and(applied_o, improved_o)
            attempted_t = np.logical_and(applied_t, improved_t)

            ratio_o.append(applied_o.mean())
            ratio_t.append(applied_t.mean())

            use_opt_o = attempted_o if use_only_improved else applied_o
            use_opt_t = attempted_t if use_only_improved else applied_t

            final_corr_o = np.where(use_opt_o, opt_corr[:, r], corr)
            final_corr_t = np.where(use_opt_t, opt_corr_t[:, r], corr_t)

            acc_o.append(final_corr_o.mean())
            acc_t.append(final_corr_t.mean())

        acc_orig_runs.append(acc_o)
        acc_trans_runs.append(acc_t)
        ratio_orig_runs.append(ratio_o)
        ratio_trans_runs.append(ratio_t)

    # Convert to numpy for CI calculation
    acc_orig_runs, acc_trans_runs = np.array(acc_orig_runs), np.array(acc_trans_runs)
    ratio_orig_runs, ratio_trans_runs = np.array(ratio_orig_runs), np.array(ratio_trans_runs)

    # ---- 4. PLOTTING ----
    def compute_ci(y):
        n = y.shape[0]
        mean = y.mean(axis=0)
        se = y.std(axis=0, ddof=1) / np.sqrt(n) if n > 1 else np.zeros_like(mean)
        return mean, mean - ci_multiplier * se, mean + ci_multiplier * se

    fig, ax1 = plt.subplots(figsize=(W, W/3.1))
    tab10 = plt.get_cmap("tab10")
    col_o, col_t, col_ro, col_rt = tab10(0), tab10(1), tab10(2), tab10(3)

    ax1.grid(True, linestyle="--", alpha=0.6)

    m_o, lo_o, hi_o = compute_ci(acc_orig_runs)
    m_t, lo_t, hi_t = compute_ci(acc_trans_runs)

    ax1.plot(plot_thresholds, m_o, label="Acc. Original", color=col_o)
    ax1.fill_between(plot_thresholds, lo_o, hi_o, alpha=0.25, color=col_o)
    ax1.plot(plot_thresholds, m_t, label="Acc. Transformed", color=col_t)
    ax1.fill_between(plot_thresholds, lo_t, hi_t, alpha=0.25, color=col_t)

    ax1.set_xlabel("Threshold (Recalculated)", fontsize=9)
    ax1.set_ylabel("Accuracy", fontsize=9)

    if plot_ratio:
        ax2 = ax1.twinx()
        rm_o, rlo_o, rhi_o = compute_ci(ratio_orig_runs)
        rm_t, rlo_t, rhi_t = compute_ci(ratio_trans_runs)
        ax2.plot(plot_thresholds, rm_o, linestyle="--", color=col_ro, label="Ratio Original")
        ax2.plot(plot_thresholds, rm_t, linestyle="--", color=col_rt, label="Ratio Transformed")
        ax2.set_ylabel(ratio_name)


    ax1.set_xlim([ql_trans, qh_trans])
    if plot_ratio:
        ax2.set_xlim([ql_trans, qh_trans])


         # ---- Combined legend ----
    lines1, labels1 = ax1.get_legend_handles_labels()
    if plot_ratio:
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax2.legend(lines1 + lines2, labels1 + labels2, loc="lower right", fontsize=8, framealpha=0.5,labelspacing=0.2,)
    else:
        ax1.legend(lines1, labels1, loc="center right", fontsize=8, framealpha=0.5,labelspacing=0.2,)

    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path,bbox_inches="tight")


    plt.show()

    print(f"[{dataset_name}] Direction: {'Decreasing' if is_decreasing else 'Increasing'}")
    print(f"Range: {ql_trans:.4f} to {qh_trans:.4f}")


In [ ]:
cache_file = default_cache_path("imagenet_tradeoff3_resumable")
results = calculate_confidences_and_cache_resumable(
    optimizer=optimizer_obj,
    best_problem=problem_to_use,
    conf_fn=conf_fn,
    test_loader=loader_in,
    test_loader_transformed=test_loader_transformed,
    device=device,
    n_runs=2,
    cache_dir=cache_file,
)
figure_path2 = os.path.join(current_path,"experiment_files","export","fig2","comp_tradeoff",dataset,transform_name)
os.makedirs(figure_path2, exist_ok=True)
plot_confidence_threshold_results(results, dataset_name=dataset, save_path=os.path.join(figure_path2, "confidence_threshold_resum2.pdf"),use_only_improved=True,relative_improvement=0.05,ci_multiplier=1.96,threshold_transform=lambda y:1/y-1)
print("Saved plot to:", os.path.join(results_dir_trade, "confidence_threshold_resum2.pdf"))

In [ ]:
results["optimized_correct_predicted_transformed"][:,1].mean()

In [ ]:
import torch
import numpy as np
import tqdm

@torch.no_grad()
def evaluate_at_threshold(
    optimizer,
    problem,
    conf_fn,
    test_loader_transformed,
    device,
    threshold: float = 0.8,
):
    all_correct = []
    total_samples = 0
    correct_count = 0
    attempted_opt = 0
    successful_opt = 0

    pbar = tqdm.tqdm(test_loader_transformed, desc=f"Evaluating @ threshold {threshold}")

    for x_test, y_test in pbar:
        x_test = x_test.to(device)
        batch_size = x_test.shape[0]

        # 1. Get baseline confidence
        conf_baseline, logits_baseline = conf_fn(x_test)
        if logits_baseline is None:
            logits_baseline = model(x_test)
        pred_baseline = torch.argmax(logits_baseline, dim=1)

        # 2. Identify samples below threshold
        below_threshold = conf_baseline < threshold

        final_pred = pred_baseline.clone()

        if below_threshold.any():
            attempted_opt += below_threshold.sum().item()

            # --- FIX: OPTIMIZE FULL BATCH TO PRESERVE STATS ---
            # Even though we only care about 'below_threshold',
            # we pass the full x_test to the optimizer so it sees
            # the same statistics as your simulation.
            p, _, _ = optimizer.optimize(problem, x_test)

            # Transform the inputs
            x_optimized_full = problem.transform_sequence.transform(x_test, p)

            # 3. Check Confidence on the optimized images
            # We only calculate confidence for the relevant ones to save compute,
            # or calculate full and mask. Calculating full is safer for indexing.
            conf_opt_full, logits_opt_full = conf_fn(x_optimized_full)
            if logits_opt_full is None:
                logits_opt_full = model(x_optimized_full)
            pred_opt_full = torch.argmax(logits_opt_full, dim=1)

            # 4. Accept optimization ONLY if confidence improved
            # logic: (Below Threshold) AND (New Conf > Old Conf)
            improved_mask = (conf_opt_full > conf_baseline) & below_threshold

            successful_opt += improved_mask.sum().item()

            # Update predictions for the improved samples
            final_pred[improved_mask] = pred_opt_full[improved_mask]

        # Compute correctness
        correct = (final_pred.cpu() == y_test).numpy()
        correct_count += correct.sum()
        total_samples += batch_size

        current_acc = correct_count / total_samples
        pbar.set_postfix({"Acc": f"{current_acc:.4f}"})

    final_acc = correct_count / total_samples
    opt_ratio = attempted_opt / total_samples
    success_ratio = successful_opt / attempted_opt if attempted_opt > 0 else 0

    print(f"\n=== Results at threshold {threshold} ===")
    print(f"Final Accuracy: {final_acc:.4f}")
    return final_acc


test_loader_transformed_shuffle= torch.utils.data.DataLoader(
    test_loader_transformed.dataset,
    batch_size=32,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

#print("")
# Run evaluation
# results_threshold = evaluate_at_threshold(
#     optimizer=di,
#     problem=problem_nn_pytorch_pretrained,
#     conf_fn=problem_nn_pytorch_pretrained.confidence_module,
#     test_loader_transformed=test_loader_transformed_shuffle,
#     device=device,
#     threshold=0.8
# )

In [ ]:
import time
import torch
from typing import Dict

def benchmark_confidence_methods(
    model,
    confidence_module,  # PCKNN or similar
    test_loader,
    num_batches: int = 15,
    device: Optional[torch.device] = None,
    warmup_batches: int = 3,
) -> Dict[str, float]:
    """
    Compare speed of energy-based confidence (from logits) vs PCKNN.

    Returns:
        Dict containing timing results and speedup factor.
    """
    if device is None:
        device = next(model.parameters()).device

    model.eval()
    confidence_module.eval()

    # Get batches
    batches = []
    for i, (data, _) in enumerate(test_loader):
        if i >= num_batches + warmup_batches:
            break
        batches.append(data.to(device))

    if len(batches) < num_batches + warmup_batches:
        raise ValueError(f"Not enough batches in loader. Got {len(batches)}, need {num_batches + warmup_batches}")

    # === Warmup phase ===
    with torch.no_grad():
        for i in range(warmup_batches):
            data = batches[i]
            # Energy warmup
            logits = model(data)
            _ = -torch.logsumexp(logits, dim=-1)
            # PCKNN warmup
            _, _ = confidence_module(data)

    # === Energy-based confidence timing ===
    torch.cuda.synchronize() if torch.cuda.is_available() else None
    energy_times = []

    with torch.no_grad():
        for i in range(warmup_batches, warmup_batches + num_batches):
            data = batches[i]

            start = time.perf_counter()
            logits = model(data)
            energy_conf = -torch.logsumexp(logits, dim=-1)
            torch.cuda.synchronize() if torch.cuda.is_available() else None
            end = time.perf_counter()

            energy_times.append(end - start)

    # === PCKNN timing ===
    torch.cuda.synchronize() if torch.cuda.is_available() else None
    pcknn_times = []

    with torch.no_grad():
        for i in range(warmup_batches, warmup_batches + num_batches):
            data = batches[i]

            start = time.perf_counter()
            conf, _ = confidence_module(data)
            torch.cuda.synchronize() if torch.cuda.is_available() else None
            end = time.perf_counter()

            pcknn_times.append(end - start)

    # === Calculate statistics ===
    energy_mean = np.mean(energy_times)
    energy_std = np.std(energy_times)
    pcknn_mean = np.mean(pcknn_times)
    pcknn_std = np.std(pcknn_times)

    # Speedup: positive if energy is faster, negative if PCKNN is faster
    speedup = pcknn_mean / energy_mean

    return {
        "energy_mean_time": energy_mean,
        "energy_std_time": energy_std,
        "energy_total_time": sum(energy_times),
        "pcknn_mean_time": pcknn_mean,
        "pcknn_std_time": pcknn_std,
        "pcknn_total_time": sum(pcknn_times),
        "speedup_factor": speedup,  # > 1 means energy is faster
        "num_batches": num_batches,
        "energy_times": energy_times,
        "pcknn_times": pcknn_times,
    }

# Usage example:
results = benchmark_confidence_methods(
    model=model,
    confidence_module=problem_nn_pytorch_pretrained.confidence_module,
    test_loader=test_loader,
    num_batches=15,
    device=device,
)

print(f"Energy-based confidence: {results['energy_mean_time']*1000:.2f} ± {results['energy_std_time']*1000:.2f} ms")
print(f"PCKNN confidence: {results['pcknn_mean_time']*1000:.2f} ± {results['pcknn_std_time']*1000:.2f} ms")
print(f"Speedup factor: {results['speedup_factor']:.2f}x")
if results['speedup_factor'] > 1:
    print(f"Energy is {results['speedup_factor']:.2f}x faster than PCKNN")
else:
    print(f"PCKNN is {1/results['speedup_factor']:.2f}x faster than Energy")

In [ ]:
import time
import torch
import numpy as np

def measure_optimizer_speed(optimizer, problem, model, test_loader, device, speedtest_batches=15):
    """
    Measures the average inference time of an optimizer on a given problem.

    Args:
        optimizer: Optimizer object with an `optimize` method.
        problem: Problem object with a `transform` method.
        model: Torch model to evaluate transformed samples.
        test_loader: DataLoader providing test batches.
        device: Torch device (cpu or cuda).
        speedtest_batches: Number of batches to time (default: 15).

    Returns:
        dict with average time per batch and per sample.
    """
    model.eval().to(device)
    torch.cuda.empty_cache()
    test_iter = iter(test_loader)

    times = []
    total_samples = 0

    with torch.no_grad():
        for batch_idx in range(speedtest_batches):
            try:
                data, _ = next(test_iter)
            except StopIteration:
                break
            data = data.to(device)
            total_samples += data.size(0)

            # Timing start
            torch.cuda.synchronize(device) if torch.cuda.is_available() else None
            start_time = time.perf_counter()

            # Optimizer inference + model evaluation
            res = optimizer.optimize(problem, data, y=None)
            best_param = res[0] if isinstance(res, tuple) and len(res) >= 1 else res
            x_trans = problem.transform(data, best_param)
            _ = model(x_trans)

            # Timing end
            torch.cuda.synchronize(device) if torch.cuda.is_available() else None
            elapsed = time.perf_counter() - start_time

            if batch_idx > 0:
                times.append(elapsed)
                print(f"[SpeedTest] Batch {batch_idx+1}: {elapsed:.4f}s")
            else:
                print(f"[SpeedTest] Batch {batch_idx+1} (warmup, ignored): {elapsed:.4f}s")

    avg_batch_time = np.mean(times)
    avg_sample_time = avg_batch_time / (total_samples / len(times))

    result = {
        "batches_tested": len(times),
        "avg_time_per_batch_sec": avg_batch_time,
        "avg_time_per_sample_sec": avg_sample_time,
    }

    return result


In [ ]:
speed_result = measure_optimizer_speed(
    optimizer=di,
    problem=problem_nn_pytorch_pretrained,
    model=model,
    test_loader=test_loader_transformed,
    device=device,
    speedtest_batches=15
)

print(speed_result)

In [ ]:
speed_result = measure_optimizer_speed(
    optimizer=di,
    problem=problem_energy_logits,
    model=model,
    test_loader=test_loader_transformed,
    device=device,
    speedtest_batches=15
)

print(speed_result)

In [ ]:
stop

In [ ]:
from typing import Optional
def visualize_canonicalization_grid(
    model,
    optimizer,
    problem,
    test_loader,
    n_rows: int = 4,
    n_cols: int = 4,
    device: str = "cuda",
    save_prefix: Optional[str] = None
):
    """
    Visualize images before and after canonicalization in a 4x4 grid.
    Green border = correct classification, Red border = incorrect.

    Args:
        save_prefix: If provided, saves {prefix}_before.pdf and {prefix}_after.pdf
    """
    model.eval()
    n_samples = n_rows * n_cols

    # Get a batch of data
    data, target = next(iter(test_loader))
    data, target = data.to(device), target.to(device)
    if len(data) < n_samples:
        data2, target2 = next(iter(test_loader))
        data = torch.cat([data, data2.to(device)], dim=0)
        target = torch.cat([target, target2.to(device)], dim=0)


    data = data[:n_samples]
    target = target[:n_samples]

    # Optimize transformations (canonicalize)
    res = optimizer.optimize(problem, data, y=target)
    params = res[0]
    x_canonicalized = problem.transform(data, params)

    # Get predictions
    with torch.no_grad():
        pred_before = model(data).argmax(dim=-1)
        pred_after = model(x_canonicalized).argmax(dim=-1)

    # unnormalize for visualization
    data = unnormalize(data)
    x_canonicalized = unnormalize(x_canonicalized)
    def add_border(img_tensor, is_correct):
        """Add colored border to image (C, H, W)"""
        color = torch.tensor([0, 1, 0] if is_correct else [1, 0, 0], device=img_tensor.device)
        bordered = img_tensor.clone()

        # Handle grayscale
        if bordered.shape[0] == 1:
            bordered = bordered.repeat(3, 1, 1)

        # Add border (2 pixel width)
        border_width = 2
        for c in range(3):
            bordered[c, :border_width, :] = color[c]
            bordered[c, -border_width:, :] = color[c]
            bordered[c, :, :border_width] = color[c]
            bordered[c, :, -border_width:] = color[c]

        return bordered

    # Create bordered images
    data_bordered = torch.stack([add_border(data[i], pred_before[i] == target[i]) for i in range(n_samples)])
    canon_bordered = torch.stack([add_border(x_canonicalized[i], pred_after[i] == target[i]) for i in range(n_samples)])

    # Create grids
    grid_before = torchvision.utils.make_grid(data_bordered.cpu(), nrow=n_cols, padding=2)
    grid_after = torchvision.utils.make_grid(canon_bordered.cpu(), nrow=n_cols, padding=2)

    # Display side-by-side
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    axes[0].imshow(grid_before.permute(1, 2, 0))
    axes[0].set_title('Before Canonicalization', fontsize=14)
    axes[0].axis('off')
    axes[1].imshow(grid_after.permute(1, 2, 0))
    axes[1].set_title('After Canonicalization', fontsize=14)
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()

    # Save separate PDFs if requested
    if save_prefix:
        # Before
        fig_before, ax_before = plt.subplots(1, 1, figsize=(8, 8))
        ax_before.imshow(grid_before.permute(1, 2, 0))
        ax_before.axis('off')
        plt.tight_layout(pad=0)
        plt.savefig(f"{save_prefix}_before.pdf", bbox_inches='tight', pad_inches=0)
        plt.close(fig_before)

        # After
        fig_after, ax_after = plt.subplots(1, 1, figsize=(8, 8))
        ax_after.imshow(grid_after.permute(1, 2, 0))
        ax_after.axis('off')
        plt.tight_layout(pad=0)
        plt.savefig(f"{save_prefix}_after.pdf", bbox_inches='tight', pad_inches=0)
        plt.close(fig_after)

        print(f"Saved: {save_prefix}_before.pdf and {save_prefix}_after.pdf")

    # Print statistics
    acc_before = (pred_before == target).float().mean().item()
    acc_after = (pred_after == target).float().mean().item()

    print(f"Accuracy - Before: {acc_before:.3f}, After: {acc_after:.3f}")

    return {"accuracy_before": acc_before, "accuracy_after": acc_after}

visualize_canonicalization_grid(
    model=model,
    optimizer=di,
    problem=problem_nn_pytorch_pretrained,
    test_loader=test_loader_transformed,
    n_rows=4,
    n_cols=4,
    device=device,
    save_prefix=savepath("canonicalization_grid").replace(".json", "")
)